# Notebook 30 — Paired Writer-Conditioned Script Geometry Audit

## Objective

Notebook 27 established a strong DINOv2-S writer-verification baseline using frozen 384-D DINOv2-S features and a small 384→144 trainable projection.

Notebook 28 showed that the standard writer projection naturally reduced linearly accessible script and text-condition information.

Notebook 29 then tested explicit script-adversarial training. Although the attached adversarial head became strongly confused, an independent writer-disjoint probe recovered substantially more script information, while writer-verification performance also declined.

Therefore, the next step is not another adversarial variant.

This notebook investigates a different question:

> Does cross-script variation exhibit a reproducible writer-conditioned geometric structure that can be estimated directly from paired handwriting observations?

This notebook is a **geometry audit**, not a new writer-verification method.

No nuisance-removal transform will be proposed unless the measured geometry supports it.

## Paired QUWI Structure

Each development writer has four pages:

- p1: Arabic, variable text
- p2: Arabic, fixed text
- p3: English, variable text
- p4: English, fixed text

This structure allows script changes to be compared while holding writer identity fixed.

For writer \(w\), define:

\[
\Delta^{var}_w = x_{w,p3} - x_{w,p1}
\]

and:

\[
\Delta^{fixed}_w = x_{w,p4} - x_{w,p2}
\]

These are same-writer English-minus-Arabic displacement vectors under matched text-condition categories.

A centroid-based displacement is also defined as:

\[
A_w = \frac{x_{w,p1} + x_{w,p2}}{2}
\]

\[
E_w = \frac{x_{w,p3} + x_{w,p4}}{2}
\]

\[
\Delta^{centroid}_w = E_w - A_w
\]

which is equivalently:

\[
\Delta^{centroid}_w
=
\frac{\Delta^{var}_w + \Delta^{fixed}_w}{2}
\]

Averaging variable and fixed pages balances the two text-condition categories within each script.

However, these displacement vectors will **not** be assumed to contain pure script information.

They may also contain:

- writer-by-script interactions,
- page-specific variation,
- residual text effects,
- representation noise.

The purpose of this notebook is to measure whether a stable shared structure exists despite these factors.

## Representations

The audit will examine two frozen representations.

### Raw DINOv2-S representation

**cached L2-normalized 384-D DINOv2-S features**

This reveals how script change is organized in the original foundation representation.

### Standard writer projection

**384-D DINOv2-S → Linear 384→144 → L2 normalization**

using the established Notebook 27 seed-42 development checkpoint.

This allows us to determine how ordinary writer metric learning changes the paired script geometry without explicit nuisance supervision.

No parameters will be trained in this notebook.

## Primary Geometry Questions

The audit will test the following questions.

### 1. Within-writer consistency

Are the variable-text and fixed-text script displacements aligned for the same writer?

That is, is:

\[
\cos(
\Delta^{var}_w,
\Delta^{fixed}_w
)
\]

systematically positive?

If these two independently constructed cross-script changes disagree strongly, a single shared script subspace would be poorly motivated.

### 2. Cross-writer coherence

Do different writers exhibit related English-minus-Arabic displacement directions?

A useful shared nuisance geometry requires more than large displacement magnitude.

The directions must show reproducible structure across writers.

### 3. Dimensional structure

Is the matrix of writer-conditioned script displacements concentrated in a relatively small number of directions?

This will be examined using singular-value and explained-energy structure.

No low-dimensionality assumption will be made in advance.

### 4. Held-out writer generalization

A script geometry estimated from the 145 fit writers will be evaluated on the 36 writer-disjoint selection writers.

Selection writers will not be used to construct the fit-derived basis.

This tests whether any discovered structure generalizes across writers rather than merely describing the training writers.

### 5. Relationship to writer geometry

The script-displacement geometry will be compared with between-writer centroid variation.

This is important because removing a direction that also carries strong writer information could damage verification.

The audit will therefore ask whether script-dominant and writer-dominant geometry are meaningfully distinguishable.

### 6. Effect of the standard writer projection

The same analyses will be repeated in the Notebook 27 projected representation.

This will determine whether ordinary writer metric learning:

- weakens script displacement,
- changes its dimensional structure,
- changes its cross-writer coherence,
- or changes its overlap with writer geometry.

## Clean Development Protocol

The same writer-disjoint development protocol is retained:

- Fit writers: **145**
- Selection writers: **36**
- Fit/selection writer overlap: **0**

The 145 fit writers will be used to estimate any geometric basis or summary statistics.

The 36 selection writers may only be used to evaluate whether the fit-derived geometry generalizes.

Selection outcomes will not be used to choose a subspace rank for a future method.

No monitor, validation, or official-test data will be accessed.

## Interpretation Boundary

A reproducible script-displacement subspace would not prove that script information is completely separable from writer identity.

Likewise, low overlap between measured script and writer geometry would not prove causal independence.

This notebook is intended only to establish whether the paired dataset structure provides sufficient geometric evidence to justify a later writer-preserving nuisance-subspace method.

If the geometry is weak, unstable across writers, or heavily entangled with writer variation, the proposed subspace direction will be rejected before building a new model.

If the geometry is strong and generalizes to unseen writers, a separate notebook may then test a controlled nuisance-subspace method.

In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [2]:
ROOT = Path.cwd().resolve()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

ROLE_PATH = (
    ROOT
    / "reports"
    / "cross_script_writer_geometry_consistency"
    / "development_internal_writer_roles_seed42.csv"
)

DINO_S_SOURCE_PATH = (
    ROOT
    / "reports"
    / "dinov2s_student_baseline"
    / "dinov2s_final_refit_aligned_features.npz"
)

BASELINE_CHECKPOINT_PATH = (
    ROOT
    / "checkpoints"
    / "dinov2s_student_baseline"
    / "dinov2s_projection_seed42_best.pt"
)

FINAL_REFIT_CHECKPOINT_PATH = (
    ROOT
    / "checkpoints"
    / "dinov2s_student_baseline"
    / "dinov2s_projection_final_refit_seed42_epoch10.pt"
)

REPORT_DIR = (
    ROOT
    / "reports"
    / "paired_writer_conditioned_script_geometry_audit"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

required_paths = {
    "development_roles": ROLE_PATH,
    "dinov2s_source": DINO_S_SOURCE_PATH,
    "notebook27_development_checkpoint": BASELINE_CHECKPOINT_PATH,
}

missing_paths = {
    name: str(path)
    for name, path in required_paths.items()
    if not path.exists()
}

if len(missing_paths) != 0:
    raise FileNotFoundError(
        json.dumps(
            missing_paths,
            indent=2,
        )
    )

role_df = pd.read_csv(
    ROLE_PATH
)

source_archive = np.load(
    DINO_S_SOURCE_PATH,
    allow_pickle=False,
)

required_archive_keys = {
    "refit_embeddings",
    "refit_filenames",
    "refit_writers",
    "refit_page_ids",
}

if not required_archive_keys.issubset(
    source_archive.files
):
    raise RuntimeError(
        "Required DINOv2-S source arrays were not found."
    )

source_embeddings = (
    source_archive[
        "refit_embeddings"
    ]
    .astype(
        np.float32,
        copy=False,
    )
)

source_filenames = (
    source_archive[
        "refit_filenames"
    ]
    .astype(str)
)

source_writers = (
    source_archive[
        "refit_writers"
    ]
    .astype(
        np.int64,
        copy=False,
    )
)

source_page_ids = (
    source_archive[
        "refit_page_ids"
    ]
    .astype(
        np.int64,
        copy=False,
    )
)

baseline_checkpoint = torch.load(
    BASELINE_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

if "internal_role" not in role_df.columns:
    raise RuntimeError(
        "Expected internal_role column was not found."
    )

source_metadata_df = pd.DataFrame(
    {
        "filename": source_filenames,
        "source_writer": source_writers,
        "page_id": source_page_ids,
        "source_index": np.arange(
            len(source_filenames),
            dtype=np.int64,
        ),
    }
)

development_role_df = (
    role_df[
        role_df[
            "internal_role"
        ].isin(
            [
                "fit",
                "selection",
            ]
        )
    ][
        [
            "filename",
            "writer",
            "internal_role",
        ]
    ]
    .copy()
)

development_role_df[
    "filename"
] = (
    development_role_df[
        "filename"
    ]
    .astype(str)
)

aligned_metadata_df = (
    development_role_df
    .merge(
        source_metadata_df,
        on="filename",
        how="inner",
        validate="one_to_one",
    )
)

aligned_metadata_df[
    "writer"
] = (
    aligned_metadata_df[
        "writer"
    ]
    .astype(int)
)

aligned_metadata_df[
    "source_writer"
] = (
    aligned_metadata_df[
        "source_writer"
    ]
    .astype(int)
)

aligned_metadata_df[
    "page_id"
] = (
    aligned_metadata_df[
        "page_id"
    ]
    .astype(int)
)

if not (
    aligned_metadata_df[
        "writer"
    ].to_numpy()
    ==
    aligned_metadata_df[
        "source_writer"
    ].to_numpy()
).all():
    raise RuntimeError(
        "Writer mismatch during DINOv2-S metadata alignment."
    )

fit_df = (
    aligned_metadata_df[
        aligned_metadata_df[
            "internal_role"
        ] == "fit"
    ]
    .copy()
)

selection_df = (
    aligned_metadata_df[
        aligned_metadata_df[
            "internal_role"
        ] == "selection"
    ]
    .copy()
)

fit_writer_set = set(
    fit_df[
        "writer"
    ]
    .unique()
    .tolist()
)

selection_writer_set = set(
    selection_df[
        "writer"
    ]
    .unique()
    .tolist()
)

fit_writer_page_counts = (
    fit_df
    .groupby(
        "writer"
    )[
        "page_id"
    ]
    .nunique()
)

selection_writer_page_counts = (
    selection_df
    .groupby(
        "writer"
    )[
        "page_id"
    ]
    .nunique()
)

fit_page_sets = (
    fit_df
    .groupby(
        "writer"
    )[
        "page_id"
    ]
    .apply(
        lambda values: tuple(
            sorted(
                values.tolist()
            )
        )
    )
)

selection_page_sets = (
    selection_df
    .groupby(
        "writer"
    )[
        "page_id"
    ]
    .apply(
        lambda values: tuple(
            sorted(
                values.tolist()
            )
        )
    )
)

source_norms = np.linalg.norm(
    source_embeddings,
    axis=1,
)

page_semantics = {
    1: {
        "script": "Arabic",
        "text_condition": "variable",
    },
    2: {
        "script": "Arabic",
        "text_condition": "fixed",
    },
    3: {
        "script": "English",
        "text_condition": "variable",
    },
    4: {
        "script": "English",
        "text_condition": "fixed",
    },
}

aligned_metadata_df[
    "script"
] = (
    aligned_metadata_df[
        "page_id"
    ]
    .map(
        {
            page_id: values[
                "script"
            ]
            for page_id, values in (
                page_semantics.items()
            )
        }
    )
)

aligned_metadata_df[
    "text_condition"
] = (
    aligned_metadata_df[
        "page_id"
    ]
    .map(
        {
            page_id: values[
                "text_condition"
            ]
            for page_id, values in (
                page_semantics.items()
            )
        }
    )
)

fit_page_id_counts = (
    fit_df[
        "page_id"
    ]
    .value_counts()
    .sort_index()
    .to_dict()
)

selection_page_id_counts = (
    selection_df[
        "page_id"
    ]
    .value_counts()
    .sort_index()
    .to_dict()
)

development_checkpoint_is_final_refit = bool(
    BASELINE_CHECKPOINT_PATH.resolve()
    == FINAL_REFIT_CHECKPOINT_PATH.resolve()
)

resource_provenance_audit = {
    "notebook": 30,
    "experiment": (
        "paired writer-conditioned script geometry audit"
    ),
    "source_representation": (
        "cached L2-normalized DINOv2-S 384-D features"
    ),
    "source_archive": str(
        DINO_S_SOURCE_PATH.relative_to(
            ROOT
        )
    ),
    "source_archive_keys": sorted(
        source_archive.files
    ),
    "source_pages": int(
        len(
            source_embeddings
        )
    ),
    "source_writers": int(
        len(
            np.unique(
                source_writers
            )
        )
    ),
    "source_feature_dimension": int(
        source_embeddings.shape[
            1
        ]
    ),
    "source_features_unit_normalized": bool(
        np.allclose(
            source_norms,
            1.0,
            rtol=0.0,
            atol=1e-5,
        )
    ),
    "source_norm_min": float(
        source_norms.min()
    ),
    "source_norm_mean": float(
        source_norms.mean()
    ),
    "source_norm_max": float(
        source_norms.max()
    ),
    "aligned_pages": int(
        len(
            aligned_metadata_df
        )
    ),
    "fit_pages": int(
        len(
            fit_df
        )
    ),
    "fit_writers": int(
        len(
            fit_writer_set
        )
    ),
    "selection_pages": int(
        len(
            selection_df
        )
    ),
    "selection_writers": int(
        len(
            selection_writer_set
        )
    ),
    "fit_selection_writer_overlap": int(
        len(
            fit_writer_set
            & selection_writer_set
        )
    ),
    "fit_page_id_counts": {
        str(key): int(value)
        for key, value in (
            fit_page_id_counts.items()
        )
    },
    "selection_page_id_counts": {
        str(key): int(value)
        for key, value in (
            selection_page_id_counts.items()
        )
    },
    "all_fit_writers_have_four_pages": bool(
        (
            fit_writer_page_counts
            == 4
        ).all()
    ),
    "all_selection_writers_have_four_pages": bool(
        (
            selection_writer_page_counts
            == 4
        ).all()
    ),
    "all_fit_writers_have_exact_page_set_1_2_3_4": bool(
        (
            fit_page_sets
            == (
                1,
                2,
                3,
                4,
            )
        ).all()
    ),
    "all_selection_writers_have_exact_page_set_1_2_3_4": bool(
        (
            selection_page_sets
            == (
                1,
                2,
                3,
                4,
            )
        ).all()
    ),
    "page_semantics": {
        str(key): value
        for key, value in (
            page_semantics.items()
        )
    },
    "development_checkpoint": str(
        BASELINE_CHECKPOINT_PATH.relative_to(
            ROOT
        )
    ),
    "development_checkpoint_epoch": int(
        baseline_checkpoint[
            "epoch"
        ]
    ),
    "development_checkpoint_training_seed": int(
        baseline_checkpoint[
            "training_seed"
        ]
    ),
    "development_checkpoint_projection_dimension": int(
        baseline_checkpoint[
            "projection_dimension"
        ]
    ),
    "development_checkpoint_fit_writers": int(
        baseline_checkpoint[
            "fit_writers"
        ]
    ),
    "development_checkpoint_selection_writers": int(
        baseline_checkpoint[
            "selection_writers"
        ]
    ),
    "development_checkpoint_source_features_l2_normalized": bool(
        baseline_checkpoint[
            "source_features_l2_normalized"
        ]
    ),
    "development_checkpoint_is_final_refit_checkpoint": bool(
        development_checkpoint_is_final_refit
    ),
    "geometry_computed": False,
    "subspace_rank_selected": False,
    "nuisance_removal_transform_applied": False,
    "parameters_trained": False,
    "selection_used_to_construct_geometry": False,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
}

with open(
    REPORT_DIR
    / "paired_script_geometry_resource_provenance_audit.json",
    "w",
) as file:
    json.dump(
        resource_provenance_audit,
        file,
        indent=2,
    )

print(
    json.dumps(
        resource_provenance_audit,
        indent=2,
    )
)

if (
    source_embeddings.shape
    != (
        724,
        384,
    )
    or len(
        np.unique(
            source_writers
        )
    )
    != 181
    or len(
        aligned_metadata_df
    )
    != 724
    or len(
        fit_df
    )
    != 580
    or len(
        fit_writer_set
    )
    != 145
    or len(
        selection_df
    )
    != 144
    or len(
        selection_writer_set
    )
    != 36
    or len(
        fit_writer_set
        & selection_writer_set
    )
    != 0
    or not (
        fit_writer_page_counts
        == 4
    ).all()
    or not (
        selection_writer_page_counts
        == 4
    ).all()
    or not (
        fit_page_sets
        == (
            1,
            2,
            3,
            4,
        )
    ).all()
    or not (
        selection_page_sets
        == (
            1,
            2,
            3,
            4,
        )
    ).all()
    or fit_page_id_counts
    != {
        1: 145,
        2: 145,
        3: 145,
        4: 145,
    }
    or selection_page_id_counts
    != {
        1: 36,
        2: 36,
        3: 36,
        4: 36,
    }
    or not np.allclose(
        source_norms,
        1.0,
        rtol=0.0,
        atol=1e-5,
    )
    or int(
        baseline_checkpoint[
            "epoch"
        ]
    )
    != 10
    or int(
        baseline_checkpoint[
            "training_seed"
        ]
    )
    != 42
    or int(
        baseline_checkpoint[
            "projection_dimension"
        ]
    )
    != 144
    or int(
        baseline_checkpoint[
            "fit_writers"
        ]
    )
    != 145
    or int(
        baseline_checkpoint[
            "selection_writers"
        ]
    )
    != 36
    or not baseline_checkpoint[
        "source_features_l2_normalized"
    ]
    or development_checkpoint_is_final_refit
):
    raise RuntimeError(
        "Notebook 30 resource/provenance audit failed."
    )

{
  "notebook": 30,
  "experiment": "paired writer-conditioned script geometry audit",
  "source_representation": "cached L2-normalized DINOv2-S 384-D features",
  "source_archive": "reports/dinov2s_student_baseline/dinov2s_final_refit_aligned_features.npz",
  "source_archive_keys": [
    "refit_embeddings",
    "refit_filenames",
    "refit_page_ids",
    "refit_writers"
  ],
  "source_pages": 724,
  "source_writers": 181,
  "source_feature_dimension": 384,
  "source_features_unit_normalized": true,
  "source_norm_min": 0.9999998807907104,
  "source_norm_mean": 1.0,
  "source_norm_max": 1.0000001192092896,
  "aligned_pages": 724,
  "fit_pages": 580,
  "fit_writers": 145,
  "selection_pages": 144,
  "selection_writers": 36,
  "fit_selection_writer_overlap": 0,
  "fit_page_id_counts": {
    "1": 145,
    "2": 145,
    "3": 145,
    "4": 145
  },
  "selection_page_id_counts": {
    "1": 36,
    "2": 36,
    "3": 36,
    "4": 36
  },
  "all_fit_writers_have_four_pages": true,
  "all_selec

In [3]:
DINO_S_FEATURE_DIM = 384
PROJECTION_DIM = 144


class DINOv2SProjectionStudent(
    nn.Module
):
    def __init__(
        self,
        input_dimension=384,
        projection_dimension=144,
    ):
        super().__init__()

        self.projection = nn.Linear(
            input_dimension,
            projection_dimension,
            bias=True,
        )

    def forward(
        self,
        features,
    ):
        features = F.normalize(
            features,
            p=2,
            dim=-1,
        )

        embeddings = self.projection(
            features
        )

        embeddings = F.normalize(
            embeddings,
            p=2,
            dim=-1,
        )

        return embeddings


fit_rows = (
    aligned_metadata_df[
        aligned_metadata_df[
            "internal_role"
        ] == "fit"
    ]
    .sort_values(
        [
            "writer",
            "page_id",
        ]
    )
    .reset_index(
        drop=True
    )
)

selection_rows = (
    aligned_metadata_df[
        aligned_metadata_df[
            "internal_role"
        ] == "selection"
    ]
    .sort_values(
        [
            "writer",
            "page_id",
        ]
    )
    .reset_index(
        drop=True
    )
)


fit_raw_flat = (
    source_embeddings[
        fit_rows[
            "source_index"
        ]
        .astype(int)
        .to_numpy()
    ]
    .astype(
        np.float32,
        copy=False,
    )
)

selection_raw_flat = (
    source_embeddings[
        selection_rows[
            "source_index"
        ]
        .astype(int)
        .to_numpy()
    ]
    .astype(
        np.float32,
        copy=False,
    )
)


projection_model = (
    DINOv2SProjectionStudent(
        input_dimension=(
            DINO_S_FEATURE_DIM
        ),
        projection_dimension=(
            PROJECTION_DIM
        ),
    )
)

projection_model.load_state_dict(
    baseline_checkpoint[
        "model_state_dict"
    ],
    strict=True,
)

projection_model.eval()


with torch.no_grad():
    fit_projected_flat = (
        projection_model(
            torch.from_numpy(
                fit_raw_flat
            ).float()
        )
        .cpu()
        .numpy()
        .astype(
            np.float32,
            copy=False,
        )
    )

    selection_projected_flat = (
        projection_model(
            torch.from_numpy(
                selection_raw_flat
            ).float()
        )
        .cpu()
        .numpy()
        .astype(
            np.float32,
            copy=False,
        )
    )


fit_writer_ids = (
    fit_rows[
        "writer"
    ]
    .astype(int)
    .drop_duplicates()
    .to_numpy(
        dtype=np.int64
    )
)

selection_writer_ids = (
    selection_rows[
        "writer"
    ]
    .astype(int)
    .drop_duplicates()
    .to_numpy(
        dtype=np.int64
    )
)


fit_raw_by_writer = np.stack(
    [
        fit_raw_flat[
            fit_rows[
                "writer"
            ]
            .astype(int)
            .to_numpy()
            == writer
        ]
        for writer in (
            fit_writer_ids
        )
    ],
    axis=0,
)

selection_raw_by_writer = np.stack(
    [
        selection_raw_flat[
            selection_rows[
                "writer"
            ]
            .astype(int)
            .to_numpy()
            == writer
        ]
        for writer in (
            selection_writer_ids
        )
    ],
    axis=0,
)


fit_projected_by_writer = np.stack(
    [
        fit_projected_flat[
            fit_rows[
                "writer"
            ]
            .astype(int)
            .to_numpy()
            == writer
        ]
        for writer in (
            fit_writer_ids
        )
    ],
    axis=0,
)

selection_projected_by_writer = np.stack(
    [
        selection_projected_flat[
            selection_rows[
                "writer"
            ]
            .astype(int)
            .to_numpy()
            == writer
        ]
        for writer in (
            selection_writer_ids
        )
    ],
    axis=0,
)


fit_page_matrix = np.stack(
    [
        fit_rows[
            fit_rows[
                "writer"
            ]
            .astype(int)
            == writer
        ][
            "page_id"
        ]
        .astype(int)
        .to_numpy()
        for writer in (
            fit_writer_ids
        )
    ],
    axis=0,
)

selection_page_matrix = np.stack(
    [
        selection_rows[
            selection_rows[
                "writer"
            ]
            .astype(int)
            == writer
        ][
            "page_id"
        ]
        .astype(int)
        .to_numpy()
        for writer in (
            selection_writer_ids
        )
    ],
    axis=0,
)


expected_page_matrix_fit = np.tile(
    np.asarray(
        [
            1,
            2,
            3,
            4,
        ],
        dtype=np.int64,
    ),
    (
        len(
            fit_writer_ids
        ),
        1,
    ),
)

expected_page_matrix_selection = np.tile(
    np.asarray(
        [
            1,
            2,
            3,
            4,
        ],
        dtype=np.int64,
    ),
    (
        len(
            selection_writer_ids
        ),
        1,
    ),
)


fit_raw_norms = np.linalg.norm(
    fit_raw_flat,
    axis=1,
)

selection_raw_norms = np.linalg.norm(
    selection_raw_flat,
    axis=1,
)

fit_projected_norms = np.linalg.norm(
    fit_projected_flat,
    axis=1,
)

selection_projected_norms = np.linalg.norm(
    selection_projected_flat,
    axis=1,
)


projection_parameter_count = int(
    sum(
        parameter.numel()
        for parameter in (
            projection_model.parameters()
        )
    )
)


REPRESENTATION_ARTIFACT_PATH = (
    REPORT_DIR
    / "paired_script_geometry_aligned_representations.npz"
)


np.savez_compressed(
    REPRESENTATION_ARTIFACT_PATH,
    fit_writer_ids=(
        fit_writer_ids
    ),
    selection_writer_ids=(
        selection_writer_ids
    ),
    fit_raw_by_writer=(
        fit_raw_by_writer
    ),
    selection_raw_by_writer=(
        selection_raw_by_writer
    ),
    fit_projected_by_writer=(
        fit_projected_by_writer
    ),
    selection_projected_by_writer=(
        selection_projected_by_writer
    ),
    fit_page_ids=(
        fit_page_matrix
    ),
    selection_page_ids=(
        selection_page_matrix
    ),
    fit_filenames=np.asarray(
        fit_rows[
            "filename"
        ]
        .astype(str)
        .to_numpy()
        .reshape(
            len(
                fit_writer_ids
            ),
            4,
        ),
        dtype=str,
    ),
    selection_filenames=np.asarray(
        selection_rows[
            "filename"
        ]
        .astype(str)
        .to_numpy()
        .reshape(
            len(
                selection_writer_ids
            ),
            4,
        ),
        dtype=str,
    ),
)


representation_alignment_audit = {
    "notebook": 30,
    "raw_representation": (
        "cached L2-normalized DINOv2-S"
    ),
    "raw_dimension": int(
        DINO_S_FEATURE_DIM
    ),
    "projected_representation": (
        "Notebook 27 seed-42 development projection"
    ),
    "projected_dimension": int(
        PROJECTION_DIM
    ),
    "projection_parameter_count": int(
        projection_parameter_count
    ),
    "projection_checkpoint_epoch": int(
        baseline_checkpoint[
            "epoch"
        ]
    ),
    "projection_checkpoint_seed": int(
        baseline_checkpoint[
            "training_seed"
        ]
    ),
    "fit_writer_count": int(
        len(
            fit_writer_ids
        )
    ),
    "selection_writer_count": int(
        len(
            selection_writer_ids
        )
    ),
    "fit_raw_flat_shape": list(
        fit_raw_flat.shape
    ),
    "selection_raw_flat_shape": list(
        selection_raw_flat.shape
    ),
    "fit_projected_flat_shape": list(
        fit_projected_flat.shape
    ),
    "selection_projected_flat_shape": list(
        selection_projected_flat.shape
    ),
    "fit_raw_writer_page_shape": list(
        fit_raw_by_writer.shape
    ),
    "selection_raw_writer_page_shape": list(
        selection_raw_by_writer.shape
    ),
    "fit_projected_writer_page_shape": list(
        fit_projected_by_writer.shape
    ),
    "selection_projected_writer_page_shape": list(
        selection_projected_by_writer.shape
    ),
    "fit_page_order_exactly_1_2_3_4": bool(
        np.array_equal(
            fit_page_matrix,
            expected_page_matrix_fit,
        )
    ),
    "selection_page_order_exactly_1_2_3_4": bool(
        np.array_equal(
            selection_page_matrix,
            expected_page_matrix_selection,
        )
    ),
    "fit_raw_unit_normalized": bool(
        np.allclose(
            fit_raw_norms,
            1.0,
            rtol=0.0,
            atol=1e-5,
        )
    ),
    "selection_raw_unit_normalized": bool(
        np.allclose(
            selection_raw_norms,
            1.0,
            rtol=0.0,
            atol=1e-5,
        )
    ),
    "fit_projected_unit_normalized": bool(
        np.allclose(
            fit_projected_norms,
            1.0,
            rtol=0.0,
            atol=1e-5,
        )
    ),
    "selection_projected_unit_normalized": bool(
        np.allclose(
            selection_projected_norms,
            1.0,
            rtol=0.0,
            atol=1e-5,
        )
    ),
    "fit_raw_norm_min": float(
        fit_raw_norms.min()
    ),
    "fit_raw_norm_max": float(
        fit_raw_norms.max()
    ),
    "selection_raw_norm_min": float(
        selection_raw_norms.min()
    ),
    "selection_raw_norm_max": float(
        selection_raw_norms.max()
    ),
    "fit_projected_norm_min": float(
        fit_projected_norms.min()
    ),
    "fit_projected_norm_max": float(
        fit_projected_norms.max()
    ),
    "selection_projected_norm_min": float(
        selection_projected_norms.min()
    ),
    "selection_projected_norm_max": float(
        selection_projected_norms.max()
    ),
    "representation_artifact": str(
        REPRESENTATION_ARTIFACT_PATH.relative_to(
            ROOT
        )
    ),
    "writer_conditioned_displacements_computed": False,
    "cross_writer_geometry_analyzed": False,
    "svd_computed": False,
    "subspace_rank_selected": False,
    "nuisance_transform_applied": False,
    "parameters_trained": False,
    "selection_used_to_construct_geometry": False,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
}


with open(
    REPORT_DIR
    / "paired_script_geometry_representation_alignment_audit.json",
    "w",
) as file:
    json.dump(
        representation_alignment_audit,
        file,
        indent=2,
    )


print(
    json.dumps(
        representation_alignment_audit,
        indent=2,
    )
)


if (
    projection_parameter_count
    != 55440
    or fit_raw_flat.shape
    != (
        580,
        384,
    )
    or selection_raw_flat.shape
    != (
        144,
        384,
    )
    or fit_projected_flat.shape
    != (
        580,
        144,
    )
    or selection_projected_flat.shape
    != (
        144,
        144,
    )
    or fit_raw_by_writer.shape
    != (
        145,
        4,
        384,
    )
    or selection_raw_by_writer.shape
    != (
        36,
        4,
        384,
    )
    or fit_projected_by_writer.shape
    != (
        145,
        4,
        144,
    )
    or selection_projected_by_writer.shape
    != (
        36,
        4,
        144,
    )
    or not np.array_equal(
        fit_page_matrix,
        expected_page_matrix_fit,
    )
    or not np.array_equal(
        selection_page_matrix,
        expected_page_matrix_selection,
    )
    or not np.allclose(
        fit_raw_norms,
        1.0,
        rtol=0.0,
        atol=1e-5,
    )
    or not np.allclose(
        selection_raw_norms,
        1.0,
        rtol=0.0,
        atol=1e-5,
    )
    or not np.allclose(
        fit_projected_norms,
        1.0,
        rtol=0.0,
        atol=1e-5,
    )
    or not np.allclose(
        selection_projected_norms,
        1.0,
        rtol=0.0,
        atol=1e-5,
    )
):
    raise RuntimeError(
        "Notebook 30 representation alignment audit failed."
    )

{
  "notebook": 30,
  "raw_representation": "cached L2-normalized DINOv2-S",
  "raw_dimension": 384,
  "projected_representation": "Notebook 27 seed-42 development projection",
  "projected_dimension": 144,
  "projection_parameter_count": 55440,
  "projection_checkpoint_epoch": 10,
  "projection_checkpoint_seed": 42,
  "fit_writer_count": 145,
  "selection_writer_count": 36,
  "fit_raw_flat_shape": [
    580,
    384
  ],
  "selection_raw_flat_shape": [
    144,
    384
  ],
  "fit_projected_flat_shape": [
    580,
    144
  ],
  "selection_projected_flat_shape": [
    144,
    144
  ],
  "fit_raw_writer_page_shape": [
    145,
    4,
    384
  ],
  "selection_raw_writer_page_shape": [
    36,
    4,
    384
  ],
  "fit_projected_writer_page_shape": [
    145,
    4,
    144
  ],
  "selection_projected_writer_page_shape": [
    36,
    4,
    144
  ],
  "fit_page_order_exactly_1_2_3_4": true,
  "selection_page_order_exactly_1_2_3_4": true,
  "fit_raw_unit_normalized": true,
  "selectio

In [4]:
def rowwise_cosine(
    left,
    right,
    epsilon=1e-12,
):
    left_norms = np.linalg.norm(
        left,
        axis=1,
    )

    right_norms = np.linalg.norm(
        right,
        axis=1,
    )

    denominator = (
        left_norms
        * right_norms
    )

    if (
        denominator
        <= epsilon
    ).any():
        raise RuntimeError(
            "Near-zero displacement encountered."
        )

    return (
        np.sum(
            left
            * right,
            axis=1,
        )
        / denominator
    )


def build_fit_script_displacements(
    writer_page_features,
):
    variable = (
        writer_page_features[
            :,
            2,
            :,
        ]
        - writer_page_features[
            :,
            0,
            :,
        ]
    )

    fixed = (
        writer_page_features[
            :,
            3,
            :,
        ]
        - writer_page_features[
            :,
            1,
            :,
        ]
    )

    arabic_centroid = (
        writer_page_features[
            :,
            0,
            :,
        ]
        + writer_page_features[
            :,
            1,
            :,
        ]
    ) / 2.0

    english_centroid = (
        writer_page_features[
            :,
            2,
            :,
        ]
        + writer_page_features[
            :,
            3,
            :,
        ]
    ) / 2.0

    centroid = (
        english_centroid
        - arabic_centroid
    )

    centroid_from_matched_pairs = (
        variable
        + fixed
    ) / 2.0

    return {
        "variable": variable,
        "fixed": fixed,
        "arabic_centroid": (
            arabic_centroid
        ),
        "english_centroid": (
            english_centroid
        ),
        "centroid": centroid,
        "centroid_from_matched_pairs": (
            centroid_from_matched_pairs
        ),
    }


fit_raw_displacements = (
    build_fit_script_displacements(
        fit_raw_by_writer
    )
)

fit_projected_displacements = (
    build_fit_script_displacements(
        fit_projected_by_writer
    )
)


raw_centroid_identity_error = float(
    np.max(
        np.abs(
            fit_raw_displacements[
                "centroid"
            ]
            - fit_raw_displacements[
                "centroid_from_matched_pairs"
            ]
        )
    )
)

projected_centroid_identity_error = float(
    np.max(
        np.abs(
            fit_projected_displacements[
                "centroid"
            ]
            - fit_projected_displacements[
                "centroid_from_matched_pairs"
            ]
        )
    )
)


raw_matched_cosines = (
    rowwise_cosine(
        fit_raw_displacements[
            "variable"
        ],
        fit_raw_displacements[
            "fixed"
        ],
    )
)

projected_matched_cosines = (
    rowwise_cosine(
        fit_projected_displacements[
            "variable"
        ],
        fit_projected_displacements[
            "fixed"
        ],
    )
)


raw_variable_norms = np.linalg.norm(
    fit_raw_displacements[
        "variable"
    ],
    axis=1,
)

raw_fixed_norms = np.linalg.norm(
    fit_raw_displacements[
        "fixed"
    ],
    axis=1,
)

projected_variable_norms = np.linalg.norm(
    fit_projected_displacements[
        "variable"
    ],
    axis=1,
)

projected_fixed_norms = np.linalg.norm(
    fit_projected_displacements[
        "fixed"
    ],
    axis=1,
)


def summarize_alignment(
    values,
):
    return {
        "count": int(
            len(
                values
            )
        ),
        "mean": float(
            np.mean(
                values
            )
        ),
        "std": float(
            np.std(
                values,
                ddof=1,
            )
        ),
        "median": float(
            np.median(
                values
            )
        ),
        "minimum": float(
            np.min(
                values
            )
        ),
        "q25": float(
            np.quantile(
                values,
                0.25,
            )
        ),
        "q75": float(
            np.quantile(
                values,
                0.75,
            )
        ),
        "maximum": float(
            np.max(
                values
            )
        ),
        "positive_count": int(
            (
                values
                > 0.0
            ).sum()
        ),
        "positive_fraction": float(
            (
                values
                > 0.0
            ).mean()
        ),
        "above_0_25_fraction": float(
            (
                values
                > 0.25
            ).mean()
        ),
        "above_0_50_fraction": float(
            (
                values
                > 0.50
            ).mean()
        ),
    }


raw_alignment_summary = (
    summarize_alignment(
        raw_matched_cosines
    )
)

projected_alignment_summary = (
    summarize_alignment(
        projected_matched_cosines
    )
)


fit_writer_alignment_df = pd.DataFrame(
    {
        "writer": (
            fit_writer_ids
        ),
        "raw_variable_fixed_cosine": (
            raw_matched_cosines
        ),
        "projected_variable_fixed_cosine": (
            projected_matched_cosines
        ),
        "raw_variable_displacement_norm": (
            raw_variable_norms
        ),
        "raw_fixed_displacement_norm": (
            raw_fixed_norms
        ),
        "projected_variable_displacement_norm": (
            projected_variable_norms
        ),
        "projected_fixed_displacement_norm": (
            projected_fixed_norms
        ),
    }
)


FIT_DISPLACEMENT_ARTIFACT_PATH = (
    REPORT_DIR
    / "paired_script_geometry_fit_displacements.npz"
)


np.savez_compressed(
    FIT_DISPLACEMENT_ARTIFACT_PATH,
    fit_writer_ids=(
        fit_writer_ids
    ),
    raw_variable=(
        fit_raw_displacements[
            "variable"
        ]
    ),
    raw_fixed=(
        fit_raw_displacements[
            "fixed"
        ]
    ),
    raw_centroid=(
        fit_raw_displacements[
            "centroid"
        ]
    ),
    raw_arabic_centroid=(
        fit_raw_displacements[
            "arabic_centroid"
        ]
    ),
    raw_english_centroid=(
        fit_raw_displacements[
            "english_centroid"
        ]
    ),
    projected_variable=(
        fit_projected_displacements[
            "variable"
        ]
    ),
    projected_fixed=(
        fit_projected_displacements[
            "fixed"
        ]
    ),
    projected_centroid=(
        fit_projected_displacements[
            "centroid"
        ]
    ),
    projected_arabic_centroid=(
        fit_projected_displacements[
            "arabic_centroid"
        ]
    ),
    projected_english_centroid=(
        fit_projected_displacements[
            "english_centroid"
        ]
    ),
)


fit_writer_alignment_df.to_csv(
    REPORT_DIR
    / "paired_script_geometry_fit_within_writer_alignment.csv",
    index=False,
)


fit_within_writer_alignment_audit = {
    "notebook": 30,
    "hypothesis": (
        "matched variable-text and fixed-text "
        "English-minus-Arabic displacements show "
        "within-writer directional consistency"
    ),
    "data_scope": (
        "145 fit writers only"
    ),
    "fit_writers": int(
        len(
            fit_writer_ids
        )
    ),
    "raw_variable_displacement_shape": list(
        fit_raw_displacements[
            "variable"
        ].shape
    ),
    "raw_fixed_displacement_shape": list(
        fit_raw_displacements[
            "fixed"
        ].shape
    ),
    "projected_variable_displacement_shape": list(
        fit_projected_displacements[
            "variable"
        ].shape
    ),
    "projected_fixed_displacement_shape": list(
        fit_projected_displacements[
            "fixed"
        ].shape
    ),
    "raw_centroid_identity_max_abs_error": float(
        raw_centroid_identity_error
    ),
    "projected_centroid_identity_max_abs_error": float(
        projected_centroid_identity_error
    ),
    "raw_matched_variable_fixed_cosine": (
        raw_alignment_summary
    ),
    "projected_matched_variable_fixed_cosine": (
        projected_alignment_summary
    ),
    "raw_variable_displacement_norm_mean": float(
        raw_variable_norms.mean()
    ),
    "raw_fixed_displacement_norm_mean": float(
        raw_fixed_norms.mean()
    ),
    "projected_variable_displacement_norm_mean": float(
        projected_variable_norms.mean()
    ),
    "projected_fixed_displacement_norm_mean": float(
        projected_fixed_norms.mean()
    ),
    "selection_analyzed": False,
    "cross_writer_coherence_analyzed": False,
    "svd_computed": False,
    "subspace_rank_selected": False,
    "nuisance_transform_applied": False,
    "parameters_trained": False,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
}


with open(
    REPORT_DIR
    / "paired_script_geometry_fit_within_writer_alignment_audit.json",
    "w",
) as file:
    json.dump(
        fit_within_writer_alignment_audit,
        file,
        indent=2,
    )


print(
    json.dumps(
        fit_within_writer_alignment_audit,
        indent=2,
    )
)


alignment_summary_df = pd.DataFrame(
    [
        {
            "representation": (
                "raw_dinov2s"
            ),
            **raw_alignment_summary,
        },
        {
            "representation": (
                "notebook27_projection"
            ),
            **projected_alignment_summary,
        },
    ]
)


print(
    "\nFit-only matched displacement alignment:"
)

print(
    alignment_summary_df
    .round(
        6
    )
    .to_string(
        index=False
    )
)


if (
    fit_raw_displacements[
        "variable"
    ].shape
    != (
        145,
        384,
    )
    or fit_raw_displacements[
        "fixed"
    ].shape
    != (
        145,
        384,
    )
    or fit_projected_displacements[
        "variable"
    ].shape
    != (
        145,
        144,
    )
    or fit_projected_displacements[
        "fixed"
    ].shape
    != (
        145,
        144,
    )
    or raw_centroid_identity_error
    > 1e-6
    or projected_centroid_identity_error
    > 1e-6
    or not np.isfinite(
        raw_matched_cosines
    ).all()
    or not np.isfinite(
        projected_matched_cosines
    ).all()
    or (
        raw_variable_norms
        <= 1e-12
    ).any()
    or (
        raw_fixed_norms
        <= 1e-12
    ).any()
    or (
        projected_variable_norms
        <= 1e-12
    ).any()
    or (
        projected_fixed_norms
        <= 1e-12
    ).any()
):
    raise RuntimeError(
        "Notebook 30 fit within-writer displacement audit failed."
    )

{
  "notebook": 30,
  "hypothesis": "matched variable-text and fixed-text English-minus-Arabic displacements show within-writer directional consistency",
  "data_scope": "145 fit writers only",
  "fit_writers": 145,
  "raw_variable_displacement_shape": [
    145,
    384
  ],
  "raw_fixed_displacement_shape": [
    145,
    384
  ],
  "projected_variable_displacement_shape": [
    145,
    144
  ],
  "projected_fixed_displacement_shape": [
    145,
    144
  ],
  "raw_centroid_identity_max_abs_error": 1.4901161193847656e-08,
  "projected_centroid_identity_max_abs_error": 2.9802322387695312e-08,
  "raw_matched_variable_fixed_cosine": {
    "count": 145,
    "mean": 0.35349714756011963,
    "std": 0.18307514488697052,
    "median": 0.3686622977256775,
    "minimum": -0.29007795453071594,
    "q25": 0.2434900850057602,
    "q75": 0.4819333553314209,
    "maximum": 0.7390922904014587,
    "positive_count": 139,
    "positive_fraction": 0.9586206896551724,
    "above_0_25_fraction": 0.73793

In [5]:
def cosine_matrix(
    left,
    right,
    epsilon=1e-12,
):
    left_norms = np.linalg.norm(
        left,
        axis=1,
        keepdims=True,
    )

    right_norms = np.linalg.norm(
        right,
        axis=1,
        keepdims=True,
    )

    if (
        left_norms
        <= epsilon
    ).any() or (
        right_norms
        <= epsilon
    ).any():
        raise RuntimeError(
            "Near-zero vector encountered."
        )

    left_unit = (
        left
        / left_norms
    )

    right_unit = (
        right
        / right_norms
    )

    return (
        left_unit
        @ right_unit.T
    )


def summarize_cross_writer_values(
    values,
):
    return {
        "count": int(
            len(
                values
            )
        ),
        "mean": float(
            np.mean(
                values
            )
        ),
        "std": float(
            np.std(
                values,
                ddof=1,
            )
        ),
        "median": float(
            np.median(
                values
            )
        ),
        "minimum": float(
            np.min(
                values
            )
        ),
        "q25": float(
            np.quantile(
                values,
                0.25,
            )
        ),
        "q75": float(
            np.quantile(
                values,
                0.75,
            )
        ),
        "maximum": float(
            np.max(
                values
            )
        ),
        "positive_fraction": float(
            (
                values
                > 0.0
            ).mean()
        ),
    }


def analyze_cross_writer_coherence(
    variable,
    fixed,
    centroid,
):
    writer_count = int(
        variable.shape[
            0
        ]
    )

    off_diagonal_mask = ~np.eye(
        writer_count,
        dtype=bool,
    )

    variable_fixed_matrix = (
        cosine_matrix(
            variable,
            fixed,
        )
    )

    matched_values = np.diag(
        variable_fixed_matrix
    )

    unmatched_values = (
        variable_fixed_matrix[
            off_diagonal_mask
        ]
    )

    centroid_matrix = (
        cosine_matrix(
            centroid,
            centroid,
        )
    )

    centroid_cross_writer_values = (
        centroid_matrix[
            off_diagonal_mask
        ]
    )

    centroid_norms = np.linalg.norm(
        centroid,
        axis=1,
        keepdims=True,
    )

    centroid_unit = (
        centroid
        / centroid_norms
    )

    mean_direction_vector = (
        centroid_unit.mean(
            axis=0
        )
    )

    resultant_length = float(
        np.linalg.norm(
            mean_direction_vector
        )
    )

    leave_one_out_alignments = []

    for writer_index in range(
        writer_count
    ):
        leave_one_out_mean = (
            centroid_unit.sum(
                axis=0
            )
            - centroid_unit[
                writer_index
            ]
        ) / (
            writer_count
            - 1
        )

        leave_one_out_norm = float(
            np.linalg.norm(
                leave_one_out_mean
            )
        )

        if leave_one_out_norm <= 1e-12:
            raise RuntimeError(
                "Near-zero leave-one-out mean direction encountered."
            )

        leave_one_out_alignment = float(
            np.dot(
                centroid_unit[
                    writer_index
                ],
                leave_one_out_mean
                / leave_one_out_norm,
            )
        )

        leave_one_out_alignments.append(
            leave_one_out_alignment
        )

    leave_one_out_alignments = np.asarray(
        leave_one_out_alignments,
        dtype=np.float64,
    )

    return {
        "matched_variable_fixed": (
            summarize_cross_writer_values(
                matched_values
            )
        ),
        "cross_writer_variable_fixed": (
            summarize_cross_writer_values(
                unmatched_values
            )
        ),
        "matched_minus_cross_writer_mean": float(
            matched_values.mean()
            - unmatched_values.mean()
        ),
        "cross_writer_centroid_pairwise": (
            summarize_cross_writer_values(
                centroid_cross_writer_values
            )
        ),
        "centroid_direction_resultant_length": float(
            resultant_length
        ),
        "centroid_leave_one_out_alignment": (
            summarize_cross_writer_values(
                leave_one_out_alignments
            )
        ),
        "matched_values": (
            matched_values
        ),
        "unmatched_values": (
            unmatched_values
        ),
        "centroid_cross_writer_values": (
            centroid_cross_writer_values
        ),
        "leave_one_out_alignments": (
            leave_one_out_alignments
        ),
    }


raw_cross_writer_result = (
    analyze_cross_writer_coherence(
        variable=(
            fit_raw_displacements[
                "variable"
            ]
        ),
        fixed=(
            fit_raw_displacements[
                "fixed"
            ]
        ),
        centroid=(
            fit_raw_displacements[
                "centroid"
            ]
        ),
    )
)


projected_cross_writer_result = (
    analyze_cross_writer_coherence(
        variable=(
            fit_projected_displacements[
                "variable"
            ]
        ),
        fixed=(
            fit_projected_displacements[
                "fixed"
            ]
        ),
        centroid=(
            fit_projected_displacements[
                "centroid"
            ]
        ),
    )
)


cross_writer_summary_rows = []

for representation, result in [
    (
        "raw_dinov2s",
        raw_cross_writer_result,
    ),
    (
        "notebook27_projection",
        projected_cross_writer_result,
    ),
]:
    cross_writer_summary_rows.append(
        {
            "representation": (
                representation
            ),
            "matched_variable_fixed_mean": float(
                result[
                    "matched_variable_fixed"
                ][
                    "mean"
                ]
            ),
            "matched_variable_fixed_median": float(
                result[
                    "matched_variable_fixed"
                ][
                    "median"
                ]
            ),
            "cross_writer_variable_fixed_mean": float(
                result[
                    "cross_writer_variable_fixed"
                ][
                    "mean"
                ]
            ),
            "cross_writer_variable_fixed_median": float(
                result[
                    "cross_writer_variable_fixed"
                ][
                    "median"
                ]
            ),
            "cross_writer_variable_fixed_positive_fraction": float(
                result[
                    "cross_writer_variable_fixed"
                ][
                    "positive_fraction"
                ]
            ),
            "matched_minus_cross_writer_mean": float(
                result[
                    "matched_minus_cross_writer_mean"
                ]
            ),
            "centroid_cross_writer_pairwise_mean": float(
                result[
                    "cross_writer_centroid_pairwise"
                ][
                    "mean"
                ]
            ),
            "centroid_cross_writer_pairwise_median": float(
                result[
                    "cross_writer_centroid_pairwise"
                ][
                    "median"
                ]
            ),
            "centroid_cross_writer_pairwise_positive_fraction": float(
                result[
                    "cross_writer_centroid_pairwise"
                ][
                    "positive_fraction"
                ]
            ),
            "centroid_direction_resultant_length": float(
                result[
                    "centroid_direction_resultant_length"
                ]
            ),
            "leave_one_out_alignment_mean": float(
                result[
                    "centroid_leave_one_out_alignment"
                ][
                    "mean"
                ]
            ),
            "leave_one_out_alignment_median": float(
                result[
                    "centroid_leave_one_out_alignment"
                ][
                    "median"
                ]
            ),
            "leave_one_out_alignment_positive_fraction": float(
                result[
                    "centroid_leave_one_out_alignment"
                ][
                    "positive_fraction"
                ]
            ),
        }
    )


cross_writer_summary_df = pd.DataFrame(
    cross_writer_summary_rows
)


cross_writer_coherence_audit = {
    "notebook": 30,
    "hypothesis": (
        "writer-conditioned English-minus-Arabic "
        "displacements contain a shared directional "
        "component across different fit writers"
    ),
    "data_scope": (
        "145 fit writers only"
    ),
    "fit_writers": 145,
    "cross_writer_variable_fixed_pairs": int(
        145
        * 144
    ),
    "raw": {
        "matched_variable_fixed": (
            raw_cross_writer_result[
                "matched_variable_fixed"
            ]
        ),
        "cross_writer_variable_fixed": (
            raw_cross_writer_result[
                "cross_writer_variable_fixed"
            ]
        ),
        "matched_minus_cross_writer_mean": float(
            raw_cross_writer_result[
                "matched_minus_cross_writer_mean"
            ]
        ),
        "cross_writer_centroid_pairwise": (
            raw_cross_writer_result[
                "cross_writer_centroid_pairwise"
            ]
        ),
        "centroid_direction_resultant_length": float(
            raw_cross_writer_result[
                "centroid_direction_resultant_length"
            ]
        ),
        "centroid_leave_one_out_alignment": (
            raw_cross_writer_result[
                "centroid_leave_one_out_alignment"
            ]
        ),
    },
    "projected": {
        "matched_variable_fixed": (
            projected_cross_writer_result[
                "matched_variable_fixed"
            ]
        ),
        "cross_writer_variable_fixed": (
            projected_cross_writer_result[
                "cross_writer_variable_fixed"
            ]
        ),
        "matched_minus_cross_writer_mean": float(
            projected_cross_writer_result[
                "matched_minus_cross_writer_mean"
            ]
        ),
        "cross_writer_centroid_pairwise": (
            projected_cross_writer_result[
                "cross_writer_centroid_pairwise"
            ]
        ),
        "centroid_direction_resultant_length": float(
            projected_cross_writer_result[
                "centroid_direction_resultant_length"
            ]
        ),
        "centroid_leave_one_out_alignment": (
            projected_cross_writer_result[
                "centroid_leave_one_out_alignment"
            ]
        ),
    },
    "selection_analyzed": False,
    "svd_computed": False,
    "subspace_rank_selected": False,
    "nuisance_transform_applied": False,
    "parameters_trained": False,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
}


with open(
    REPORT_DIR
    / "paired_script_geometry_fit_cross_writer_coherence_audit.json",
    "w",
) as file:
    json.dump(
        cross_writer_coherence_audit,
        file,
        indent=2,
    )


cross_writer_summary_df.to_csv(
    REPORT_DIR
    / "paired_script_geometry_fit_cross_writer_coherence_summary.csv",
    index=False,
)


print(
    json.dumps(
        cross_writer_coherence_audit,
        indent=2,
    )
)

print(
    "\nFit-only cross-writer coherence:"
)

print(
    cross_writer_summary_df
    .round(
        6
    )
    .to_string(
        index=False
    )
)


if (
    len(
        raw_cross_writer_result[
            "unmatched_values"
        ]
    )
    != 145
    * 144
    or len(
        projected_cross_writer_result[
            "unmatched_values"
        ]
    )
    != 145
    * 144
    or not np.isfinite(
        cross_writer_summary_df[
            [
                "matched_variable_fixed_mean",
                "cross_writer_variable_fixed_mean",
                "centroid_cross_writer_pairwise_mean",
                "centroid_direction_resultant_length",
                "leave_one_out_alignment_mean",
            ]
        ]
        .to_numpy()
    ).all()
):
    raise RuntimeError(
        "Notebook 30 fit cross-writer coherence audit failed."
    )

{
  "notebook": 30,
  "hypothesis": "writer-conditioned English-minus-Arabic displacements contain a shared directional component across different fit writers",
  "data_scope": "145 fit writers only",
  "fit_writers": 145,
  "cross_writer_variable_fixed_pairs": 20880,
  "raw": {
    "matched_variable_fixed": {
      "count": 145,
      "mean": 0.35349711775779724,
      "std": 0.18307514488697052,
      "median": 0.3686622381210327,
      "minimum": -0.29007795453071594,
      "q25": 0.24349012970924377,
      "q75": 0.4819333553314209,
      "maximum": 0.739092230796814,
      "positive_fraction": 0.9586206896551724
    },
    "cross_writer_variable_fixed": {
      "count": 20880,
      "mean": 0.15752336382865906,
      "std": 0.17214013636112213,
      "median": 0.16084912419319153,
      "minimum": -0.4784688353538513,
      "q25": 0.042871929705142975,
      "q75": 0.27703166007995605,
      "maximum": 0.7291529774665833,
      "positive_fraction": 0.8179597701149425
    },
    "m

In [6]:
def normalize_rows(
    matrix,
    epsilon=1e-12,
):
    norms = np.linalg.norm(
        matrix,
        axis=1,
        keepdims=True,
    )

    if (
        norms
        <= epsilon
    ).any():
        raise RuntimeError(
            "Near-zero vector encountered during row normalization."
        )

    return (
        matrix
        / norms
    )


def rank_for_energy_fraction(
    cumulative_energy,
    target,
):
    return int(
        np.searchsorted(
            cumulative_energy,
            target,
            side="left",
        )
        + 1
    )


def compute_svd_profile(
    matrix,
    center,
):
    original_matrix = (
        np.asarray(
            matrix,
            dtype=np.float64,
        )
    )

    original_mean = (
        original_matrix.mean(
            axis=0
        )
    )

    if center:
        analyzed_matrix = (
            original_matrix
            - original_mean[
                None,
                :
            ]
        )
    else:
        analyzed_matrix = (
            original_matrix.copy()
        )

    _, singular_values, right_vectors = (
        np.linalg.svd(
            analyzed_matrix,
            full_matrices=False,
        )
    )

    singular_energy = (
        singular_values
        ** 2
    )

    total_energy = float(
        singular_energy.sum()
    )

    if total_energy <= 0.0:
        raise RuntimeError(
            "SVD energy is zero."
        )

    normalized_energy = (
        singular_energy
        / total_energy
    )

    cumulative_energy = np.cumsum(
        normalized_energy
    )

    positive_energy = (
        normalized_energy[
            normalized_energy
            > 0.0
        ]
    )

    entropy_effective_rank = float(
        np.exp(
            -np.sum(
                positive_energy
                * np.log(
                    positive_energy
                )
            )
        )
    )

    participation_ratio = float(
        1.0
        / np.sum(
            normalized_energy
            ** 2
        )
    )

    mean_vector_norm = float(
        np.linalg.norm(
            original_mean
        )
    )

    original_total_squared_norm = float(
        np.sum(
            original_matrix
            ** 2
        )
    )

    mean_component_energy_fraction = float(
        original_matrix.shape[
            0
        ]
        * (
            mean_vector_norm
            ** 2
        )
        / original_total_squared_norm
    )

    if mean_vector_norm > 1e-12:
        mean_unit = (
            original_mean
            / mean_vector_norm
        )

        top_direction_mean_alignment = float(
            abs(
                np.dot(
                    right_vectors[
                        0
                    ],
                    mean_unit,
                )
            )
        )
    else:
        top_direction_mean_alignment = (
            float(
                "nan"
            )
        )

    def cumulative_at_rank(
        rank,
    ):
        usable_rank = min(
            int(
                rank
            ),
            len(
                cumulative_energy
            ),
        )

        return float(
            cumulative_energy[
                usable_rank
                - 1
            ]
        )

    return {
        "sample_count": int(
            original_matrix.shape[
                0
            ]
        ),
        "dimension": int(
            original_matrix.shape[
                1
            ]
        ),
        "centered": bool(
            center
        ),
        "matrix_rank": int(
            np.linalg.matrix_rank(
                analyzed_matrix
            )
        ),
        "mean_vector_norm": float(
            mean_vector_norm
        ),
        "mean_component_energy_fraction": float(
            mean_component_energy_fraction
        ),
        "top_direction_mean_alignment": float(
            top_direction_mean_alignment
        ),
        "top1_energy_fraction": float(
            normalized_energy[
                0
            ]
        ),
        "top2_cumulative_energy": (
            cumulative_at_rank(
                2
            )
        ),
        "top5_cumulative_energy": (
            cumulative_at_rank(
                5
            )
        ),
        "top10_cumulative_energy": (
            cumulative_at_rank(
                10
            )
        ),
        "top20_cumulative_energy": (
            cumulative_at_rank(
                20
            )
        ),
        "top50_cumulative_energy": (
            cumulative_at_rank(
                50
            )
        ),
        "rank_for_50_percent_energy": (
            rank_for_energy_fraction(
                cumulative_energy,
                0.50,
            )
        ),
        "rank_for_75_percent_energy": (
            rank_for_energy_fraction(
                cumulative_energy,
                0.75,
            )
        ),
        "rank_for_90_percent_energy": (
            rank_for_energy_fraction(
                cumulative_energy,
                0.90,
            )
        ),
        "rank_for_95_percent_energy": (
            rank_for_energy_fraction(
                cumulative_energy,
                0.95,
            )
        ),
        "entropy_effective_rank": float(
            entropy_effective_rank
        ),
        "participation_ratio": float(
            participation_ratio
        ),
        "singular_values": (
            singular_values
        ),
        "normalized_energy": (
            normalized_energy
        ),
        "cumulative_energy": (
            cumulative_energy
        ),
    }


raw_centroid_displacements = (
    fit_raw_displacements[
        "centroid"
    ]
)

projected_centroid_displacements = (
    fit_projected_displacements[
        "centroid"
    ]
)

raw_centroid_directions = (
    normalize_rows(
        raw_centroid_displacements
    )
)

projected_centroid_directions = (
    normalize_rows(
        projected_centroid_displacements
    )
)


svd_profiles = {
    "raw_displacement_uncentered": (
        compute_svd_profile(
            raw_centroid_displacements,
            center=False,
        )
    ),
    "raw_displacement_centered": (
        compute_svd_profile(
            raw_centroid_displacements,
            center=True,
        )
    ),
    "raw_direction_uncentered": (
        compute_svd_profile(
            raw_centroid_directions,
            center=False,
        )
    ),
    "raw_direction_centered": (
        compute_svd_profile(
            raw_centroid_directions,
            center=True,
        )
    ),
    "projected_displacement_uncentered": (
        compute_svd_profile(
            projected_centroid_displacements,
            center=False,
        )
    ),
    "projected_displacement_centered": (
        compute_svd_profile(
            projected_centroid_displacements,
            center=True,
        )
    ),
    "projected_direction_uncentered": (
        compute_svd_profile(
            projected_centroid_directions,
            center=False,
        )
    ),
    "projected_direction_centered": (
        compute_svd_profile(
            projected_centroid_directions,
            center=True,
        )
    ),
}


svd_summary_rows = []

for name, profile in (
    svd_profiles.items()
):
    representation = (
        "raw_dinov2s"
        if name.startswith(
            "raw_"
        )
        else "notebook27_projection"
    )

    vector_type = (
        "unit_direction"
        if "direction" in name
        else "displacement"
    )

    svd_summary_rows.append(
        {
            "profile": name,
            "representation": (
                representation
            ),
            "vector_type": (
                vector_type
            ),
            "centered": bool(
                profile[
                    "centered"
                ]
            ),
            "matrix_rank": int(
                profile[
                    "matrix_rank"
                ]
            ),
            "mean_vector_norm": float(
                profile[
                    "mean_vector_norm"
                ]
            ),
            "mean_component_energy_fraction": float(
                profile[
                    "mean_component_energy_fraction"
                ]
            ),
            "top_direction_mean_alignment": float(
                profile[
                    "top_direction_mean_alignment"
                ]
            ),
            "top1_energy_fraction": float(
                profile[
                    "top1_energy_fraction"
                ]
            ),
            "top5_cumulative_energy": float(
                profile[
                    "top5_cumulative_energy"
                ]
            ),
            "top10_cumulative_energy": float(
                profile[
                    "top10_cumulative_energy"
                ]
            ),
            "top20_cumulative_energy": float(
                profile[
                    "top20_cumulative_energy"
                ]
            ),
            "rank_for_50_percent_energy": int(
                profile[
                    "rank_for_50_percent_energy"
                ]
            ),
            "rank_for_75_percent_energy": int(
                profile[
                    "rank_for_75_percent_energy"
                ]
            ),
            "rank_for_90_percent_energy": int(
                profile[
                    "rank_for_90_percent_energy"
                ]
            ),
            "rank_for_95_percent_energy": int(
                profile[
                    "rank_for_95_percent_energy"
                ]
            ),
            "entropy_effective_rank": float(
                profile[
                    "entropy_effective_rank"
                ]
            ),
            "participation_ratio": float(
                profile[
                    "participation_ratio"
                ]
            ),
        }
    )


svd_summary_df = pd.DataFrame(
    svd_summary_rows
)


SVD_ARTIFACT_PATH = (
    REPORT_DIR
    / "paired_script_geometry_fit_centroid_svd_spectra.npz"
)


np.savez_compressed(
    SVD_ARTIFACT_PATH,
    raw_displacement_uncentered_singular_values=(
        svd_profiles[
            "raw_displacement_uncentered"
        ][
            "singular_values"
        ]
    ),
    raw_displacement_centered_singular_values=(
        svd_profiles[
            "raw_displacement_centered"
        ][
            "singular_values"
        ]
    ),
    raw_direction_uncentered_singular_values=(
        svd_profiles[
            "raw_direction_uncentered"
        ][
            "singular_values"
        ]
    ),
    raw_direction_centered_singular_values=(
        svd_profiles[
            "raw_direction_centered"
        ][
            "singular_values"
        ]
    ),
    projected_displacement_uncentered_singular_values=(
        svd_profiles[
            "projected_displacement_uncentered"
        ][
            "singular_values"
        ]
    ),
    projected_displacement_centered_singular_values=(
        svd_profiles[
            "projected_displacement_centered"
        ][
            "singular_values"
        ]
    ),
    projected_direction_uncentered_singular_values=(
        svd_profiles[
            "projected_direction_uncentered"
        ][
            "singular_values"
        ]
    ),
    projected_direction_centered_singular_values=(
        svd_profiles[
            "projected_direction_centered"
        ][
            "singular_values"
        ]
    ),
)


svd_summary_df.to_csv(
    REPORT_DIR
    / "paired_script_geometry_fit_centroid_svd_summary.csv",
    index=False,
)


def profile_for_json(
    profile,
):
    return {
        key: value
        for key, value in (
            profile.items()
        )
        if key not in {
            "singular_values",
            "normalized_energy",
            "cumulative_energy",
        }
    }


svd_dimensional_structure_audit = {
    "notebook": 30,
    "hypothesis": (
        "fit-writer centroid script displacements "
        "show concentrated dimensional structure"
    ),
    "data_scope": (
        "145 fit writers only"
    ),
    "primary_vector": (
        "English centroid minus Arabic centroid"
    ),
    "raw_displacement_uncentered": (
        profile_for_json(
            svd_profiles[
                "raw_displacement_uncentered"
            ]
        )
    ),
    "raw_displacement_centered": (
        profile_for_json(
            svd_profiles[
                "raw_displacement_centered"
            ]
        )
    ),
    "raw_direction_uncentered": (
        profile_for_json(
            svd_profiles[
                "raw_direction_uncentered"
            ]
        )
    ),
    "raw_direction_centered": (
        profile_for_json(
            svd_profiles[
                "raw_direction_centered"
            ]
        )
    ),
    "projected_displacement_uncentered": (
        profile_for_json(
            svd_profiles[
                "projected_displacement_uncentered"
            ]
        )
    ),
    "projected_displacement_centered": (
        profile_for_json(
            svd_profiles[
                "projected_displacement_centered"
            ]
        )
    ),
    "projected_direction_uncentered": (
        profile_for_json(
            svd_profiles[
                "projected_direction_uncentered"
            ]
        )
    ),
    "projected_direction_centered": (
        profile_for_json(
            svd_profiles[
                "projected_direction_centered"
            ]
        )
    ),
    "low_dimensionality_claimed": False,
    "null_model_tested": False,
    "selection_analyzed": False,
    "subspace_rank_selected": False,
    "nuisance_transform_applied": False,
    "parameters_trained": False,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
}


with open(
    REPORT_DIR
    / "paired_script_geometry_fit_svd_dimensional_structure_audit.json",
    "w",
) as file:
    json.dump(
        svd_dimensional_structure_audit,
        file,
        indent=2,
    )


print(
    json.dumps(
        svd_dimensional_structure_audit,
        indent=2,
    )
)

print(
    "\nFit-only centroid script-displacement SVD summary:"
)

print(
    svd_summary_df[
        [
            "profile",
            "top1_energy_fraction",
            "top5_cumulative_energy",
            "top10_cumulative_energy",
            "top20_cumulative_energy",
            "rank_for_50_percent_energy",
            "rank_for_75_percent_energy",
            "rank_for_90_percent_energy",
            "rank_for_95_percent_energy",
            "entropy_effective_rank",
            "participation_ratio",
            "mean_component_energy_fraction",
            "top_direction_mean_alignment",
        ]
    ]
    .round(
        6
    )
    .to_string(
        index=False
    )
)


if (
    raw_centroid_displacements.shape
    != (
        145,
        384,
    )
    or projected_centroid_displacements.shape
    != (
        145,
        144,
    )
    or len(
        svd_summary_df
    )
    != 8
    or not np.isfinite(
        svd_summary_df[
            [
                "top1_energy_fraction",
                "top5_cumulative_energy",
                "top10_cumulative_energy",
                "top20_cumulative_energy",
                "rank_for_50_percent_energy",
                "rank_for_75_percent_energy",
                "rank_for_90_percent_energy",
                "rank_for_95_percent_energy",
                "entropy_effective_rank",
                "participation_ratio",
                "mean_component_energy_fraction",
            ]
        ]
        .to_numpy()
    ).all()
):
    raise RuntimeError(
        "Notebook 30 fit SVD dimensional-structure audit failed."
    )

{
  "notebook": 30,
  "hypothesis": "fit-writer centroid script displacements show concentrated dimensional structure",
  "data_scope": "145 fit writers only",
  "primary_vector": "English centroid minus Arabic centroid",
  "raw_displacement_uncentered": {
    "sample_count": 145,
    "dimension": 384,
    "centered": false,
    "matrix_rank": 145,
    "mean_vector_norm": 0.19641902169453282,
    "mean_component_energy_fraction": 0.2793282542870593,
    "top_direction_mean_alignment": 0.9899246443002843,
    "top1_energy_fraction": 0.3231586167304145,
    "top2_cumulative_energy": 0.41618473000452505,
    "top5_cumulative_energy": 0.5777208655779296,
    "top10_cumulative_energy": 0.7227779057313632,
    "top20_cumulative_energy": 0.844116429240711,
    "top50_cumulative_energy": 0.951296358595565,
    "rank_for_50_percent_energy": 4,
    "rank_for_75_percent_energy": 12,
    "rank_for_90_percent_energy": 31,
    "rank_for_95_percent_energy": 50,
    "entropy_effective_rank": 21.022328

In [7]:
NULL_RANDOM_SEED = 42
NULL_TRIALS = 250


def random_derangement(
    size,
    rng,
):
    reference = np.arange(
        size
    )

    while True:
        permutation = rng.permutation(
            size
        )

        if np.all(
            permutation
            != reference
        ):
            return permutation


def compact_spectral_metrics(
    matrix,
):
    matrix = np.asarray(
        matrix,
        dtype=np.float64,
    )

    row_norms = np.linalg.norm(
        matrix,
        axis=1,
        keepdims=True,
    )

    if (
        row_norms
        <= 1e-12
    ).any():
        raise RuntimeError(
            "Near-zero displacement encountered."
        )

    unit_directions = (
        matrix
        / row_norms
    )

    direction_resultant_length = float(
        np.linalg.norm(
            unit_directions.mean(
                axis=0
            )
        )
    )

    mean_vector = matrix.mean(
        axis=0
    )

    metric_result = {
        "direction_resultant_length": float(
            direction_resultant_length
        )
    }

    for label, analyzed_matrix in [
        (
            "uncentered",
            matrix,
        ),
        (
            "centered",
            matrix
            - mean_vector[
                None,
                :
            ],
        ),
    ]:
        singular_values = np.linalg.svd(
            analyzed_matrix,
            full_matrices=False,
            compute_uv=False,
        )

        energy = (
            singular_values
            ** 2
        )

        normalized_energy = (
            energy
            / energy.sum()
        )

        cumulative_energy = np.cumsum(
            normalized_energy
        )

        rank90 = int(
            np.searchsorted(
                cumulative_energy,
                0.90,
                side="left",
            )
            + 1
        )

        participation_ratio = float(
            1.0
            / np.sum(
                normalized_energy
                ** 2
            )
        )

        top5_index = min(
            5,
            len(
                cumulative_energy
            ),
        ) - 1

        metric_result[
            f"{label}_top1_energy_fraction"
        ] = float(
            normalized_energy[
                0
            ]
        )

        metric_result[
            f"{label}_top5_cumulative_energy"
        ] = float(
            cumulative_energy[
                top5_index
            ]
        )

        metric_result[
            f"{label}_rank_for_90_percent_energy"
        ] = int(
            rank90
        )

        metric_result[
            f"{label}_participation_ratio"
        ] = float(
            participation_ratio
        )

    return metric_result


raw_arabic_centroids = (
    fit_raw_displacements[
        "arabic_centroid"
    ]
)

raw_english_centroids = (
    fit_raw_displacements[
        "english_centroid"
    ]
)

projected_arabic_centroids = (
    fit_projected_displacements[
        "arabic_centroid"
    ]
)

projected_english_centroids = (
    fit_projected_displacements[
        "english_centroid"
    ]
)


observed_raw_null_metrics = (
    compact_spectral_metrics(
        raw_english_centroids
        - raw_arabic_centroids
    )
)

observed_projected_null_metrics = (
    compact_spectral_metrics(
        projected_english_centroids
        - projected_arabic_centroids
    )
)


if not np.isclose(
    observed_raw_null_metrics[
        "uncentered_top1_energy_fraction"
    ],
    svd_profiles[
        "raw_displacement_uncentered"
    ][
        "top1_energy_fraction"
    ],
    rtol=0.0,
    atol=1e-10,
):
    raise RuntimeError(
        "Observed raw spectrum does not reproduce Cell 6."
    )


if not np.isclose(
    observed_projected_null_metrics[
        "centered_top1_energy_fraction"
    ],
    svd_profiles[
        "projected_displacement_centered"
    ][
        "top1_energy_fraction"
    ],
    rtol=0.0,
    atol=1e-10,
):
    raise RuntimeError(
        "Observed projected spectrum does not reproduce Cell 6."
    )


rng = np.random.default_rng(
    NULL_RANDOM_SEED
)

null_trial_rows = []


for trial in range(
    1,
    NULL_TRIALS
    + 1,
):
    permutation = (
        random_derangement(
            len(
                fit_writer_ids
            ),
            rng,
        )
    )

    raw_null_displacements = (
        raw_english_centroids[
            permutation
        ]
        - raw_arabic_centroids
    )

    projected_null_displacements = (
        projected_english_centroids[
            permutation
        ]
        - projected_arabic_centroids
    )

    raw_metrics = (
        compact_spectral_metrics(
            raw_null_displacements
        )
    )

    projected_metrics = (
        compact_spectral_metrics(
            projected_null_displacements
        )
    )

    trial_row = {
        "trial": int(
            trial
        ),
        "fixed_points": int(
            np.sum(
                permutation
                == np.arange(
                    len(
                        permutation
                    )
                )
            )
        ),
    }

    for key, value in (
        raw_metrics.items()
    ):
        trial_row[
            f"raw_{key}"
        ] = value

    for key, value in (
        projected_metrics.items()
    ):
        trial_row[
            f"projected_{key}"
        ] = value

    null_trial_rows.append(
        trial_row
    )


null_trial_df = pd.DataFrame(
    null_trial_rows
)


metric_directions = {
    "direction_resultant_length": (
        "higher"
    ),
    "uncentered_top1_energy_fraction": (
        "higher"
    ),
    "uncentered_top5_cumulative_energy": (
        "higher"
    ),
    "centered_top1_energy_fraction": (
        "higher"
    ),
    "centered_top5_cumulative_energy": (
        "higher"
    ),
    "centered_rank_for_90_percent_energy": (
        "lower"
    ),
    "centered_participation_ratio": (
        "lower"
    ),
}


null_summary_rows = []


for representation, observed_metrics in [
    (
        "raw_dinov2s",
        observed_raw_null_metrics,
    ),
    (
        "notebook27_projection",
        observed_projected_null_metrics,
    ),
]:
    column_prefix = (
        "raw_"
        if representation
        == "raw_dinov2s"
        else "projected_"
    )

    for metric, extremeness_direction in (
        metric_directions.items()
    ):
        null_values = (
            null_trial_df[
                column_prefix
                + metric
            ]
            .to_numpy(
                dtype=np.float64
            )
        )

        observed_value = float(
            observed_metrics[
                metric
            ]
        )

        null_mean = float(
            null_values.mean()
        )

        null_std = float(
            null_values.std(
                ddof=1
            )
        )

        if (
            extremeness_direction
            == "higher"
        ):
            extreme_count = int(
                (
                    null_values
                    >= observed_value
                ).sum()
            )
        else:
            extreme_count = int(
                (
                    null_values
                    <= observed_value
                ).sum()
            )

        empirical_tail_fraction = float(
            (
                extreme_count
                + 1
            )
            / (
                NULL_TRIALS
                + 1
            )
        )

        standardized_difference = (
            float(
                (
                    observed_value
                    - null_mean
                )
                / null_std
            )
            if null_std
            > 0.0
            else float(
                "nan"
            )
        )

        null_summary_rows.append(
            {
                "representation": (
                    representation
                ),
                "metric": (
                    metric
                ),
                "concentration_extreme_direction": (
                    extremeness_direction
                ),
                "observed": float(
                    observed_value
                ),
                "null_mean": float(
                    null_mean
                ),
                "null_std": float(
                    null_std
                ),
                "null_q025": float(
                    np.quantile(
                        null_values,
                        0.025,
                    )
                ),
                "null_median": float(
                    np.median(
                        null_values
                    )
                ),
                "null_q975": float(
                    np.quantile(
                        null_values,
                        0.975,
                    )
                ),
                "observed_minus_null_mean": float(
                    observed_value
                    - null_mean
                ),
                "standardized_difference": float(
                    standardized_difference
                ),
                "empirical_extreme_count": int(
                    extreme_count
                ),
                "empirical_tail_fraction": float(
                    empirical_tail_fraction
                ),
            }
        )


null_summary_df = pd.DataFrame(
    null_summary_rows
)


NULL_TRIAL_PATH = (
    REPORT_DIR
    / "paired_script_geometry_fit_writer_pairing_null_trials.csv"
)

NULL_SUMMARY_PATH = (
    REPORT_DIR
    / "paired_script_geometry_fit_writer_pairing_null_summary.csv"
)


null_trial_df.to_csv(
    NULL_TRIAL_PATH,
    index=False,
)

null_summary_df.to_csv(
    NULL_SUMMARY_PATH,
    index=False,
)


writer_pairing_null_audit = {
    "notebook": 30,
    "null_question": (
        "is same-writer English-minus-Arabic "
        "centroid geometry more directionally or "
        "spectrally structured than mismatched-writer "
        "Arabic-English pairings?"
    ),
    "data_scope": (
        "145 fit writers only"
    ),
    "null_construction": (
        "English writer centroids are randomly "
        "deranged relative to Arabic writer centroids"
    ),
    "null_trials": int(
        NULL_TRIALS
    ),
    "random_seed": int(
        NULL_RANDOM_SEED
    ),
    "all_null_trials_have_zero_fixed_points": bool(
        (
            null_trial_df[
                "fixed_points"
            ]
            == 0
        ).all()
    ),
    "arabic_and_english_marginals_preserved": True,
    "same_writer_pairing_destroyed": True,
    "raw_observed_metrics": {
        key: (
            int(value)
            if isinstance(
                value,
                (
                    int,
                    np.integer,
                ),
            )
            else float(
                value
            )
        )
        for key, value in (
            observed_raw_null_metrics.items()
        )
    },
    "projected_observed_metrics": {
        key: (
            int(value)
            if isinstance(
                value,
                (
                    int,
                    np.integer,
                ),
            )
            else float(
                value
            )
        )
        for key, value in (
            observed_projected_null_metrics.items()
        )
    },
    "empirical_tail_fraction_is_formal_significance_claim": False,
    "low_dimensionality_claimed": False,
    "writer_conditioned_specificity_claimed": False,
    "selection_analyzed": False,
    "subspace_rank_selected": False,
    "nuisance_transform_applied": False,
    "parameters_trained": False,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
}


with open(
    REPORT_DIR
    / "paired_script_geometry_fit_writer_pairing_null_audit.json",
    "w",
) as file:
    json.dump(
        writer_pairing_null_audit,
        file,
        indent=2,
    )


print(
    json.dumps(
        writer_pairing_null_audit,
        indent=2,
    )
)

print(
    "\nWriter-pairing null comparison:"
)

print(
    null_summary_df[
        [
            "representation",
            "metric",
            "observed",
            "null_mean",
            "null_q025",
            "null_median",
            "null_q975",
            "standardized_difference",
            "empirical_tail_fraction",
        ]
    ]
    .round(
        6
    )
    .to_string(
        index=False
    )
)


if (
    len(
        null_trial_df
    )
    != NULL_TRIALS
    or not (
        null_trial_df[
            "fixed_points"
        ]
        == 0
    ).all()
    or not np.isfinite(
        null_summary_df[
            [
                "observed",
                "null_mean",
                "null_std",
                "null_q025",
                "null_median",
                "null_q975",
                "empirical_tail_fraction",
            ]
        ]
        .to_numpy()
    ).all()
):
    raise RuntimeError(
        "Notebook 30 writer-pairing null audit failed."
    )

{
  "notebook": 30,
  "null_question": "is same-writer English-minus-Arabic centroid geometry more directionally or spectrally structured than mismatched-writer Arabic-English pairings?",
  "data_scope": "145 fit writers only",
  "null_construction": "English writer centroids are randomly deranged relative to Arabic writer centroids",
  "null_trials": 250,
  "random_seed": 42,
  "all_null_trials_have_zero_fixed_points": true,
  "arabic_and_english_marginals_preserved": true,
  "same_writer_pairing_destroyed": true,
  "raw_observed_metrics": {
    "direction_resultant_length": 0.5309424311599318,
    "uncentered_top1_energy_fraction": 0.32315861673041446,
    "uncentered_top5_cumulative_energy": 0.5777208655779295,
    "uncentered_rank_for_90_percent_energy": 31,
    "uncentered_participation_ratio": 7.789140805798266,
    "centered_top1_energy_fraction": 0.13706190824312042,
    "centered_top5_cumulative_energy": 0.45087203976152895,
    "centered_rank_for_90_percent_energy": 39,
    "

In [8]:
SELECTION_NULL_TRIALS = 250
SELECTION_NULL_RANDOM_SEED = 42


def cosine_to_fixed_direction(
    vectors,
    direction,
    epsilon=1e-12,
):
    vectors = np.asarray(
        vectors,
        dtype=np.float64,
    )

    direction = np.asarray(
        direction,
        dtype=np.float64,
    )

    vector_norms = np.linalg.norm(
        vectors,
        axis=1,
    )

    direction_norm = float(
        np.linalg.norm(
            direction
        )
    )

    if (
        vector_norms
        <= epsilon
    ).any() or direction_norm <= epsilon:
        raise RuntimeError(
            "Near-zero vector encountered."
        )

    direction_unit = (
        direction
        / direction_norm
    )

    return (
        vectors
        @ direction_unit
        / vector_norms
    )


def summarize_alignment_values(
    values,
):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    return {
        "count": int(
            len(
                values
            )
        ),
        "mean": float(
            values.mean()
        ),
        "std": float(
            values.std(
                ddof=1
            )
        ),
        "median": float(
            np.median(
                values
            )
        ),
        "minimum": float(
            values.min()
        ),
        "q25": float(
            np.quantile(
                values,
                0.25,
            )
        ),
        "q75": float(
            np.quantile(
                values,
                0.75,
            )
        ),
        "maximum": float(
            values.max()
        ),
        "positive_count": int(
            (
                values
                > 0.0
            ).sum()
        ),
        "positive_fraction": float(
            (
                values
                > 0.0
            ).mean()
        ),
    }


selection_raw_displacements = (
    build_fit_script_displacements(
        selection_raw_by_writer
    )
)

selection_projected_displacements = (
    build_fit_script_displacements(
        selection_projected_by_writer
    )
)


fit_raw_mean_script_direction = (
    fit_raw_displacements[
        "centroid"
    ]
    .mean(
        axis=0
    )
)

fit_projected_mean_script_direction = (
    fit_projected_displacements[
        "centroid"
    ]
    .mean(
        axis=0
    )
)


selection_raw_fit_direction_alignment = (
    cosine_to_fixed_direction(
        selection_raw_displacements[
            "centroid"
        ],
        fit_raw_mean_script_direction,
    )
)

selection_projected_fit_direction_alignment = (
    cosine_to_fixed_direction(
        selection_projected_displacements[
            "centroid"
        ],
        fit_projected_mean_script_direction,
    )
)


selection_raw_matched_alignment = (
    rowwise_cosine(
        selection_raw_displacements[
            "variable"
        ],
        selection_raw_displacements[
            "fixed"
        ],
    )
)

selection_projected_matched_alignment = (
    rowwise_cosine(
        selection_projected_displacements[
            "variable"
        ],
        selection_projected_displacements[
            "fixed"
        ],
    )
)


selection_raw_centroid_unit = (
    normalize_rows(
        selection_raw_displacements[
            "centroid"
        ]
    )
)

selection_projected_centroid_unit = (
    normalize_rows(
        selection_projected_displacements[
            "centroid"
        ]
    )
)


selection_raw_resultant_length = float(
    np.linalg.norm(
        selection_raw_centroid_unit.mean(
            axis=0
        )
    )
)

selection_projected_resultant_length = float(
    np.linalg.norm(
        selection_projected_centroid_unit.mean(
            axis=0
        )
    )
)


selection_rng = np.random.default_rng(
    SELECTION_NULL_RANDOM_SEED
)


selection_null_rows = []


for trial in range(
    1,
    SELECTION_NULL_TRIALS
    + 1,
):
    permutation = (
        random_derangement(
            len(
                selection_writer_ids
            ),
            selection_rng,
        )
    )

    raw_null_centroid_displacements = (
        selection_raw_displacements[
            "english_centroid"
        ][
            permutation
        ]
        - selection_raw_displacements[
            "arabic_centroid"
        ]
    )

    projected_null_centroid_displacements = (
        selection_projected_displacements[
            "english_centroid"
        ][
            permutation
        ]
        - selection_projected_displacements[
            "arabic_centroid"
        ]
    )

    raw_null_fit_direction_alignment = (
        cosine_to_fixed_direction(
            raw_null_centroid_displacements,
            fit_raw_mean_script_direction,
        )
    )

    projected_null_fit_direction_alignment = (
        cosine_to_fixed_direction(
            projected_null_centroid_displacements,
            fit_projected_mean_script_direction,
        )
    )

    raw_null_unit = (
        normalize_rows(
            raw_null_centroid_displacements
        )
    )

    projected_null_unit = (
        normalize_rows(
            projected_null_centroid_displacements
        )
    )

    selection_null_rows.append(
        {
            "trial": int(
                trial
            ),
            "fixed_points": int(
                np.sum(
                    permutation
                    == np.arange(
                        len(
                            permutation
                        )
                    )
                )
            ),
            "raw_mean_alignment_to_fit_direction": float(
                raw_null_fit_direction_alignment.mean()
            ),
            "raw_median_alignment_to_fit_direction": float(
                np.median(
                    raw_null_fit_direction_alignment
                )
            ),
            "raw_positive_fraction_to_fit_direction": float(
                (
                    raw_null_fit_direction_alignment
                    > 0.0
                ).mean()
            ),
            "raw_resultant_length": float(
                np.linalg.norm(
                    raw_null_unit.mean(
                        axis=0
                    )
                )
            ),
            "projected_mean_alignment_to_fit_direction": float(
                projected_null_fit_direction_alignment.mean()
            ),
            "projected_median_alignment_to_fit_direction": float(
                np.median(
                    projected_null_fit_direction_alignment
                )
            ),
            "projected_positive_fraction_to_fit_direction": float(
                (
                    projected_null_fit_direction_alignment
                    > 0.0
                ).mean()
            ),
            "projected_resultant_length": float(
                np.linalg.norm(
                    projected_null_unit.mean(
                        axis=0
                    )
                )
            ),
        }
    )


selection_null_df = pd.DataFrame(
    selection_null_rows
)


selection_observed_metrics = {
    "raw_mean_alignment_to_fit_direction": float(
        selection_raw_fit_direction_alignment.mean()
    ),
    "raw_median_alignment_to_fit_direction": float(
        np.median(
            selection_raw_fit_direction_alignment
        )
    ),
    "raw_positive_fraction_to_fit_direction": float(
        (
            selection_raw_fit_direction_alignment
            > 0.0
        ).mean()
    ),
    "raw_resultant_length": float(
        selection_raw_resultant_length
    ),
    "projected_mean_alignment_to_fit_direction": float(
        selection_projected_fit_direction_alignment.mean()
    ),
    "projected_median_alignment_to_fit_direction": float(
        np.median(
            selection_projected_fit_direction_alignment
        )
    ),
    "projected_positive_fraction_to_fit_direction": float(
        (
            selection_projected_fit_direction_alignment
            > 0.0
        ).mean()
    ),
    "projected_resultant_length": float(
        selection_projected_resultant_length
    ),
}


selection_null_comparison_rows = []


for metric, observed_value in (
    selection_observed_metrics.items()
):
    null_values = (
        selection_null_df[
            metric
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    null_mean = float(
        null_values.mean()
    )

    null_std = float(
        null_values.std(
            ddof=1
        )
    )

    extreme_count = int(
        (
            null_values
            >= observed_value
        ).sum()
    )

    empirical_tail_fraction = float(
        (
            extreme_count
            + 1
        )
        / (
            SELECTION_NULL_TRIALS
            + 1
        )
    )

    standardized_difference = (
        float(
            (
                observed_value
                - null_mean
            )
            / null_std
        )
        if null_std
        > 0.0
        else float(
            "nan"
        )
    )

    selection_null_comparison_rows.append(
        {
            "metric": (
                metric
            ),
            "observed": float(
                observed_value
            ),
            "null_mean": float(
                null_mean
            ),
            "null_std": float(
                null_std
            ),
            "null_q025": float(
                np.quantile(
                    null_values,
                    0.025,
                )
            ),
            "null_median": float(
                np.median(
                    null_values
                )
            ),
            "null_q975": float(
                np.quantile(
                    null_values,
                    0.975,
                )
            ),
            "standardized_difference": float(
                standardized_difference
            ),
            "empirical_tail_fraction": float(
                empirical_tail_fraction
            ),
        }
    )


selection_null_comparison_df = (
    pd.DataFrame(
        selection_null_comparison_rows
    )
)


selection_generalization_summary = {
    "notebook": 30,
    "question": (
        "does the fit-derived mean English-minus-Arabic "
        "direction generalize to unseen writers?"
    ),
    "fit_writers_used_to_construct_direction": 145,
    "selection_writers": 36,
    "fit_selection_writer_overlap": 0,
    "raw_fit_mean_direction_norm": float(
        np.linalg.norm(
            fit_raw_mean_script_direction
        )
    ),
    "projected_fit_mean_direction_norm": float(
        np.linalg.norm(
            fit_projected_mean_script_direction
        )
    ),
    "raw_selection_alignment_to_fit_direction": (
        summarize_alignment_values(
            selection_raw_fit_direction_alignment
        )
    ),
    "projected_selection_alignment_to_fit_direction": (
        summarize_alignment_values(
            selection_projected_fit_direction_alignment
        )
    ),
    "raw_selection_matched_variable_fixed_alignment": (
        summarize_alignment_values(
            selection_raw_matched_alignment
        )
    ),
    "projected_selection_matched_variable_fixed_alignment": (
        summarize_alignment_values(
            selection_projected_matched_alignment
        )
    ),
    "raw_selection_resultant_length": float(
        selection_raw_resultant_length
    ),
    "projected_selection_resultant_length": float(
        selection_projected_resultant_length
    ),
    "selection_pairing_null_trials": int(
        SELECTION_NULL_TRIALS
    ),
    "selection_pairing_null_random_seed": int(
        SELECTION_NULL_RANDOM_SEED
    ),
    "all_null_trials_have_zero_fixed_points": bool(
        (
            selection_null_df[
                "fixed_points"
            ]
            == 0
        ).all()
    ),
    "fit_direction_frozen_before_selection_analysis": True,
    "selection_used_to_choose_direction": False,
    "selection_used_to_choose_rank": False,
    "subspace_rank_selected": False,
    "nuisance_transform_applied": False,
    "parameters_trained": False,
    "empirical_tail_fraction_is_formal_significance_claim": False,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
}


selection_null_df.to_csv(
    REPORT_DIR
    / "paired_script_geometry_selection_pairing_null_trials.csv",
    index=False,
)


selection_null_comparison_df.to_csv(
    REPORT_DIR
    / "paired_script_geometry_selection_pairing_null_comparison.csv",
    index=False,
)


with open(
    REPORT_DIR
    / "paired_script_geometry_selection_generalization_audit.json",
    "w",
) as file:
    json.dump(
        selection_generalization_summary,
        file,
        indent=2,
    )


print(
    json.dumps(
        selection_generalization_summary,
        indent=2,
    )
)

print(
    "\nHeld-out selection pairing-null comparison:"
)

print(
    selection_null_comparison_df
    .round(
        6
    )
    .to_string(
        index=False
    )
)


if (
    selection_raw_displacements[
        "centroid"
    ].shape
    != (
        36,
        384,
    )
    or selection_projected_displacements[
        "centroid"
    ].shape
    != (
        36,
        144,
    )
    or len(
        selection_null_df
    )
    != SELECTION_NULL_TRIALS
    or not (
        selection_null_df[
            "fixed_points"
        ]
        == 0
    ).all()
    or not np.isfinite(
        selection_null_comparison_df[
            [
                "observed",
                "null_mean",
                "null_std",
                "null_q025",
                "null_median",
                "null_q975",
                "empirical_tail_fraction",
            ]
        ]
        .to_numpy()
    ).all()
):
    raise RuntimeError(
        "Notebook 30 held-out geometry generalization audit failed."
    )

{
  "notebook": 30,
  "question": "does the fit-derived mean English-minus-Arabic direction generalize to unseen writers?",
  "fit_writers_used_to_construct_direction": 145,
  "selection_writers": 36,
  "fit_selection_writer_overlap": 0,
  "raw_fit_mean_direction_norm": 0.19641901552677155,
  "projected_fit_mean_direction_norm": 0.08417856693267822,
  "raw_selection_alignment_to_fit_direction": {
    "count": 36,
    "mean": 0.524339564329646,
    "std": 0.12251099837296735,
    "median": 0.5219448158858353,
    "minimum": 0.21982634200035964,
    "q25": 0.44434896767609294,
    "q75": 0.6240512705501003,
    "maximum": 0.7140685712136292,
    "positive_count": 36,
    "positive_fraction": 1.0
  },
  "projected_selection_alignment_to_fit_direction": {
    "count": 36,
    "mean": 0.020431343447498328,
    "std": 0.228422587295437,
    "median": 0.010531828714156567,
    "minimum": -0.36701383217982336,
    "q25": -0.12119882428332308,
    "q75": 0.18052766776636436,
    "maximum": 0.47

In [9]:
WRITER_GEOMETRY_NULL_TRIALS = 5000
WRITER_GEOMETRY_NULL_SEED = 42
WRITER_SUBSPACE_RANKS = [
    1,
    5,
    10,
    20,
    50,
]


def analyze_script_writer_overlap(
    writer_page_features,
    script_displacements,
    fit_script_direction,
    null_trials,
    null_seed,
):
    writer_centroids = (
        writer_page_features.mean(
            axis=1
        )
        .astype(
            np.float64,
            copy=False,
        )
    )

    centered_writer_centroids = (
        writer_centroids
        - writer_centroids.mean(
            axis=0,
            keepdims=True,
        )
    )

    direction = np.asarray(
        fit_script_direction,
        dtype=np.float64,
    )

    direction_norm = float(
        np.linalg.norm(
            direction
        )
    )

    if direction_norm <= 1e-12:
        raise RuntimeError(
            "Near-zero script direction encountered."
        )

    direction_unit = (
        direction
        / direction_norm
    )

    writer_direction_coordinates = (
        centered_writer_centroids
        @ direction_unit
    )

    script_direction_coordinates = (
        np.asarray(
            script_displacements,
            dtype=np.float64,
        )
        @ direction_unit
    )

    writer_direction_energy = float(
        np.sum(
            writer_direction_coordinates
            ** 2
        )
    )

    total_writer_energy = float(
        np.sum(
            centered_writer_centroids
            ** 2
        )
    )

    writer_energy_fraction_along_script_direction = float(
        writer_direction_energy
        / total_writer_energy
    )

    writer_direction_std = float(
        np.std(
            writer_direction_coordinates,
            ddof=1,
        )
    )

    script_projection_mean = float(
        np.mean(
            script_direction_coordinates
        )
    )

    script_projection_median = float(
        np.median(
            script_direction_coordinates
        )
    )

    script_projection_std = float(
        np.std(
            script_direction_coordinates,
            ddof=1,
        )
    )

    script_projection_positive_fraction = float(
        (
            script_direction_coordinates
            > 0.0
        ).mean()
    )

    script_shift_to_writer_std_ratio = float(
        script_projection_mean
        / writer_direction_std
    )

    _, writer_singular_values, writer_right_vectors = (
        np.linalg.svd(
            centered_writer_centroids,
            full_matrices=False,
        )
    )

    writer_energy = (
        writer_singular_values
        ** 2
    )

    writer_energy_fraction = (
        writer_energy
        / writer_energy.sum()
    )

    writer_cumulative_energy = np.cumsum(
        writer_energy_fraction
    )

    valid_ranks = [
        rank
        for rank in (
            WRITER_SUBSPACE_RANKS
        )
        if rank
        <= writer_right_vectors.shape[
            0
        ]
    ]

    observed_subspace_overlap = {}

    for rank in (
        valid_ranks
    ):
        basis = (
            writer_right_vectors[
                :rank,
                :
            ]
        )

        squared_overlap = float(
            np.sum(
                (
                    basis
                    @ direction_unit
                )
                ** 2
            )
        )

        observed_subspace_overlap[
            rank
        ] = (
            squared_overlap
        )

    rng = np.random.default_rng(
        null_seed
    )

    random_directions = (
        rng.normal(
            size=(
                null_trials,
                centered_writer_centroids.shape[
                    1
                ],
            )
        )
    )

    random_directions = (
        random_directions
        / np.linalg.norm(
            random_directions,
            axis=1,
            keepdims=True,
        )
    )

    random_writer_coordinates = (
        centered_writer_centroids
        @ random_directions.T
    )

    random_writer_energy_fractions = (
        np.sum(
            random_writer_coordinates
            ** 2,
            axis=0,
        )
        / total_writer_energy
    )

    directional_energy_extreme_count = int(
        (
            random_writer_energy_fractions
            >= writer_energy_fraction_along_script_direction
        ).sum()
    )

    directional_energy_upper_tail = float(
        (
            directional_energy_extreme_count
            + 1
        )
        / (
            null_trials
            + 1
        )
    )

    directional_energy_null_mean = float(
        random_writer_energy_fractions.mean()
    )

    directional_energy_null_std = float(
        random_writer_energy_fractions.std(
            ddof=1
        )
    )

    directional_energy_standardized_difference = float(
        (
            writer_energy_fraction_along_script_direction
            - directional_energy_null_mean
        )
        / directional_energy_null_std
    )

    subspace_rows = []

    for rank in (
        valid_ranks
    ):
        basis = (
            writer_right_vectors[
                :rank,
                :
            ]
        )

        random_overlap = np.sum(
            (
                random_directions
                @ basis.T
            )
            ** 2,
            axis=1,
        )

        observed_overlap = float(
            observed_subspace_overlap[
                rank
            ]
        )

        upper_tail_count = int(
            (
                random_overlap
                >= observed_overlap
            ).sum()
        )

        upper_tail_fraction = float(
            (
                upper_tail_count
                + 1
            )
            / (
                null_trials
                + 1
            )
        )

        random_overlap_mean = float(
            random_overlap.mean()
        )

        random_overlap_std = float(
            random_overlap.std(
                ddof=1
            )
        )

        standardized_difference = float(
            (
                observed_overlap
                - random_overlap_mean
            )
            / random_overlap_std
        )

        subspace_rows.append(
            {
                "rank": int(
                    rank
                ),
                "writer_variance_cumulative_fraction": float(
                    writer_cumulative_energy[
                        rank
                        - 1
                    ]
                ),
                "script_direction_squared_overlap": float(
                    observed_overlap
                ),
                "random_direction_overlap_mean": float(
                    random_overlap_mean
                ),
                "random_direction_overlap_std": float(
                    random_overlap_std
                ),
                "random_direction_overlap_q025": float(
                    np.quantile(
                        random_overlap,
                        0.025,
                    )
                ),
                "random_direction_overlap_median": float(
                    np.median(
                        random_overlap
                    )
                ),
                "random_direction_overlap_q975": float(
                    np.quantile(
                        random_overlap,
                        0.975,
                    )
                ),
                "standardized_difference": float(
                    standardized_difference
                ),
                "empirical_upper_tail_fraction": float(
                    upper_tail_fraction
                ),
            }
        )

    return {
        "writer_centroids": (
            writer_centroids
        ),
        "centered_writer_centroids": (
            centered_writer_centroids
        ),
        "script_direction_unit": (
            direction_unit
        ),
        "writer_direction_coordinates": (
            writer_direction_coordinates
        ),
        "script_direction_coordinates": (
            script_direction_coordinates
        ),
        "writer_energy_fraction_along_script_direction": float(
            writer_energy_fraction_along_script_direction
        ),
        "writer_direction_std": float(
            writer_direction_std
        ),
        "script_projection_mean": float(
            script_projection_mean
        ),
        "script_projection_median": float(
            script_projection_median
        ),
        "script_projection_std": float(
            script_projection_std
        ),
        "script_projection_positive_fraction": float(
            script_projection_positive_fraction
        ),
        "script_shift_to_writer_std_ratio": float(
            script_shift_to_writer_std_ratio
        ),
        "writer_direction_energy_null_mean": float(
            directional_energy_null_mean
        ),
        "writer_direction_energy_null_std": float(
            directional_energy_null_std
        ),
        "writer_direction_energy_null_q025": float(
            np.quantile(
                random_writer_energy_fractions,
                0.025,
            )
        ),
        "writer_direction_energy_null_median": float(
            np.median(
                random_writer_energy_fractions
            )
        ),
        "writer_direction_energy_null_q975": float(
            np.quantile(
                random_writer_energy_fractions,
                0.975,
            )
        ),
        "writer_direction_energy_standardized_difference": float(
            directional_energy_standardized_difference
        ),
        "writer_direction_energy_empirical_upper_tail_fraction": float(
            directional_energy_upper_tail
        ),
        "subspace_rows": (
            subspace_rows
        ),
    }


raw_writer_overlap_result = (
    analyze_script_writer_overlap(
        writer_page_features=(
            fit_raw_by_writer
        ),
        script_displacements=(
            fit_raw_displacements[
                "centroid"
            ]
        ),
        fit_script_direction=(
            fit_raw_mean_script_direction
        ),
        null_trials=(
            WRITER_GEOMETRY_NULL_TRIALS
        ),
        null_seed=(
            WRITER_GEOMETRY_NULL_SEED
        ),
    )
)


projected_writer_overlap_result = (
    analyze_script_writer_overlap(
        writer_page_features=(
            fit_projected_by_writer
        ),
        script_displacements=(
            fit_projected_displacements[
                "centroid"
            ]
        ),
        fit_script_direction=(
            fit_projected_mean_script_direction
        ),
        null_trials=(
            WRITER_GEOMETRY_NULL_TRIALS
        ),
        null_seed=(
            WRITER_GEOMETRY_NULL_SEED
            + 1
        ),
    )
)


writer_overlap_summary_df = pd.DataFrame(
    [
        {
            "representation": (
                "raw_dinov2s"
            ),
            "writer_energy_fraction_along_script_direction": (
                raw_writer_overlap_result[
                    "writer_energy_fraction_along_script_direction"
                ]
            ),
            "random_direction_energy_mean": (
                raw_writer_overlap_result[
                    "writer_direction_energy_null_mean"
                ]
            ),
            "random_direction_energy_q975": (
                raw_writer_overlap_result[
                    "writer_direction_energy_null_q975"
                ]
            ),
            "energy_standardized_difference": (
                raw_writer_overlap_result[
                    "writer_direction_energy_standardized_difference"
                ]
            ),
            "energy_empirical_upper_tail_fraction": (
                raw_writer_overlap_result[
                    "writer_direction_energy_empirical_upper_tail_fraction"
                ]
            ),
            "writer_direction_std": (
                raw_writer_overlap_result[
                    "writer_direction_std"
                ]
            ),
            "script_projection_mean": (
                raw_writer_overlap_result[
                    "script_projection_mean"
                ]
            ),
            "script_projection_median": (
                raw_writer_overlap_result[
                    "script_projection_median"
                ]
            ),
            "script_projection_positive_fraction": (
                raw_writer_overlap_result[
                    "script_projection_positive_fraction"
                ]
            ),
            "script_shift_to_writer_std_ratio": (
                raw_writer_overlap_result[
                    "script_shift_to_writer_std_ratio"
                ]
            ),
        },
        {
            "representation": (
                "notebook27_projection"
            ),
            "writer_energy_fraction_along_script_direction": (
                projected_writer_overlap_result[
                    "writer_energy_fraction_along_script_direction"
                ]
            ),
            "random_direction_energy_mean": (
                projected_writer_overlap_result[
                    "writer_direction_energy_null_mean"
                ]
            ),
            "random_direction_energy_q975": (
                projected_writer_overlap_result[
                    "writer_direction_energy_null_q975"
                ]
            ),
            "energy_standardized_difference": (
                projected_writer_overlap_result[
                    "writer_direction_energy_standardized_difference"
                ]
            ),
            "energy_empirical_upper_tail_fraction": (
                projected_writer_overlap_result[
                    "writer_direction_energy_empirical_upper_tail_fraction"
                ]
            ),
            "writer_direction_std": (
                projected_writer_overlap_result[
                    "writer_direction_std"
                ]
            ),
            "script_projection_mean": (
                projected_writer_overlap_result[
                    "script_projection_mean"
                ]
            ),
            "script_projection_median": (
                projected_writer_overlap_result[
                    "script_projection_median"
                ]
            ),
            "script_projection_positive_fraction": (
                projected_writer_overlap_result[
                    "script_projection_positive_fraction"
                ]
            ),
            "script_shift_to_writer_std_ratio": (
                projected_writer_overlap_result[
                    "script_shift_to_writer_std_ratio"
                ]
            ),
        },
    ]
)


writer_subspace_overlap_df = pd.concat(
    [
        pd.DataFrame(
            raw_writer_overlap_result[
                "subspace_rows"
            ]
        ).assign(
            representation=(
                "raw_dinov2s"
            )
        ),
        pd.DataFrame(
            projected_writer_overlap_result[
                "subspace_rows"
            ]
        ).assign(
            representation=(
                "notebook27_projection"
            )
        ),
    ],
    ignore_index=True,
)


writer_geometry_overlap_audit = {
    "notebook": 30,
    "question": (
        "how strongly does the fit-derived mean "
        "script-shift direction overlap with "
        "between-writer centroid geometry?"
    ),
    "data_scope": (
        "145 fit writers only"
    ),
    "writer_centroid_definition": (
        "mean of all four pages per writer"
    ),
    "script_direction_definition": (
        "mean same-writer English-centroid minus "
        "Arabic-centroid displacement"
    ),
    "raw": {
        "writer_energy_fraction_along_script_direction": float(
            raw_writer_overlap_result[
                "writer_energy_fraction_along_script_direction"
            ]
        ),
        "writer_direction_std": float(
            raw_writer_overlap_result[
                "writer_direction_std"
            ]
        ),
        "script_projection_mean": float(
            raw_writer_overlap_result[
                "script_projection_mean"
            ]
        ),
        "script_projection_median": float(
            raw_writer_overlap_result[
                "script_projection_median"
            ]
        ),
        "script_projection_positive_fraction": float(
            raw_writer_overlap_result[
                "script_projection_positive_fraction"
            ]
        ),
        "script_shift_to_writer_std_ratio": float(
            raw_writer_overlap_result[
                "script_shift_to_writer_std_ratio"
            ]
        ),
        "random_direction_energy_mean": float(
            raw_writer_overlap_result[
                "writer_direction_energy_null_mean"
            ]
        ),
        "random_direction_energy_q975": float(
            raw_writer_overlap_result[
                "writer_direction_energy_null_q975"
            ]
        ),
        "energy_standardized_difference": float(
            raw_writer_overlap_result[
                "writer_direction_energy_standardized_difference"
            ]
        ),
        "energy_empirical_upper_tail_fraction": float(
            raw_writer_overlap_result[
                "writer_direction_energy_empirical_upper_tail_fraction"
            ]
        ),
    },
    "projected": {
        "writer_energy_fraction_along_script_direction": float(
            projected_writer_overlap_result[
                "writer_energy_fraction_along_script_direction"
            ]
        ),
        "writer_direction_std": float(
            projected_writer_overlap_result[
                "writer_direction_std"
            ]
        ),
        "script_projection_mean": float(
            projected_writer_overlap_result[
                "script_projection_mean"
            ]
        ),
        "script_projection_median": float(
            projected_writer_overlap_result[
                "script_projection_median"
            ]
        ),
        "script_projection_positive_fraction": float(
            projected_writer_overlap_result[
                "script_projection_positive_fraction"
            ]
        ),
        "script_shift_to_writer_std_ratio": float(
            projected_writer_overlap_result[
                "script_shift_to_writer_std_ratio"
            ]
        ),
        "random_direction_energy_mean": float(
            projected_writer_overlap_result[
                "writer_direction_energy_null_mean"
            ]
        ),
        "random_direction_energy_q975": float(
            projected_writer_overlap_result[
                "writer_direction_energy_null_q975"
            ]
        ),
        "energy_standardized_difference": float(
            projected_writer_overlap_result[
                "writer_direction_energy_standardized_difference"
            ]
        ),
        "energy_empirical_upper_tail_fraction": float(
            projected_writer_overlap_result[
                "writer_direction_energy_empirical_upper_tail_fraction"
            ]
        ),
    },
    "random_direction_null_trials": int(
        WRITER_GEOMETRY_NULL_TRIALS
    ),
    "random_direction_null_seed": int(
        WRITER_GEOMETRY_NULL_SEED
    ),
    "subspace_ranks_are_descriptive_only": True,
    "writer_preserving_removal_supported": False,
    "nuisance_transform_applied": False,
    "verification_evaluated_after_removal": False,
    "subspace_rank_selected": False,
    "parameters_trained": False,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
}


writer_overlap_summary_df.to_csv(
    REPORT_DIR
    / "paired_script_geometry_fit_writer_overlap_summary.csv",
    index=False,
)


writer_subspace_overlap_df.to_csv(
    REPORT_DIR
    / "paired_script_geometry_fit_writer_subspace_overlap.csv",
    index=False,
)


with open(
    REPORT_DIR
    / "paired_script_geometry_fit_writer_overlap_audit.json",
    "w",
) as file:
    json.dump(
        writer_geometry_overlap_audit,
        file,
        indent=2,
    )


print(
    json.dumps(
        writer_geometry_overlap_audit,
        indent=2,
    )
)

print(
    "\nScript direction versus writer geometry:"
)

print(
    writer_overlap_summary_df
    .round(
        6
    )
    .to_string(
        index=False
    )
)

print(
    "\nScript direction overlap with leading writer subspaces:"
)

print(
    writer_subspace_overlap_df[
        [
            "representation",
            "rank",
            "writer_variance_cumulative_fraction",
            "script_direction_squared_overlap",
            "random_direction_overlap_mean",
            "random_direction_overlap_q975",
            "standardized_difference",
            "empirical_upper_tail_fraction",
        ]
    ]
    .round(
        6
    )
    .to_string(
        index=False
    )
)


if (
    raw_writer_overlap_result[
        "writer_centroids"
    ].shape
    != (
        145,
        384,
    )
    or projected_writer_overlap_result[
        "writer_centroids"
    ].shape
    != (
        145,
        144,
    )
    or len(
        writer_overlap_summary_df
    )
    != 2
    or not np.isfinite(
        writer_overlap_summary_df[
            [
                "writer_energy_fraction_along_script_direction",
                "random_direction_energy_mean",
                "random_direction_energy_q975",
                "energy_standardized_difference",
                "energy_empirical_upper_tail_fraction",
                "writer_direction_std",
                "script_projection_mean",
                "script_projection_median",
                "script_projection_positive_fraction",
                "script_shift_to_writer_std_ratio",
            ]
        ]
        .to_numpy()
    ).all()
    or not np.isfinite(
        writer_subspace_overlap_df[
            [
                "writer_variance_cumulative_fraction",
                "script_direction_squared_overlap",
                "random_direction_overlap_mean",
                "random_direction_overlap_q975",
                "standardized_difference",
                "empirical_upper_tail_fraction",
            ]
        ]
        .to_numpy()
    ).all()
):
    raise RuntimeError(
        "Notebook 30 writer-geometry overlap audit failed."
    )

{
  "notebook": 30,
  "question": "how strongly does the fit-derived mean script-shift direction overlap with between-writer centroid geometry?",
  "data_scope": "145 fit writers only",
  "writer_centroid_definition": "mean of all four pages per writer",
  "script_direction_definition": "mean same-writer English-centroid minus Arabic-centroid displacement",
  "raw": {
    "writer_energy_fraction_along_script_direction": 0.05452933271817767,
    "writer_direction_std": 0.052908319990020866,
    "script_projection_mean": 0.19641902169453035,
    "script_projection_median": 0.19214497452712204,
    "script_projection_positive_fraction": 0.993103448275862,
    "script_shift_to_writer_std_ratio": 3.712441100597738,
    "random_direction_energy_mean": 0.0025856148456102576,
    "random_direction_energy_q975": 0.005470601723911859,
    "energy_standardized_difference": 46.43704425399083,
    "energy_empirical_upper_tail_fraction": 0.0001999600079984003
  },
  "projected": {
    "writer_energy

In [10]:
raw_selection_mean_alignment_row = (
    selection_null_comparison_df[
        selection_null_comparison_df[
            "metric"
        ]
        == "raw_mean_alignment_to_fit_direction"
    ]
    .iloc[
        0
    ]
)

raw_selection_resultant_row = (
    selection_null_comparison_df[
        selection_null_comparison_df[
            "metric"
        ]
        == "raw_resultant_length"
    ]
    .iloc[
        0
    ]
)

raw_centered_top5_null_row = (
    null_summary_df[
        (
            null_summary_df[
                "representation"
            ]
            == "raw_dinov2s"
        )
        & (
            null_summary_df[
                "metric"
            ]
            == "centered_top5_cumulative_energy"
        )
    ]
    .iloc[
        0
    ]
)

projected_centered_top5_null_row = (
    null_summary_df[
        (
            null_summary_df[
                "representation"
            ]
            == "notebook27_projection"
        )
        & (
            null_summary_df[
                "metric"
            ]
            == "centered_top5_cumulative_energy"
        )
    ]
    .iloc[
        0
    ]
)

raw_centered_rank90_null_row = (
    null_summary_df[
        (
            null_summary_df[
                "representation"
            ]
            == "raw_dinov2s"
        )
        & (
            null_summary_df[
                "metric"
            ]
            == "centered_rank_for_90_percent_energy"
        )
    ]
    .iloc[
        0
    ]
)

projected_centered_rank90_null_row = (
    null_summary_df[
        (
            null_summary_df[
                "representation"
            ]
            == "notebook27_projection"
        )
        & (
            null_summary_df[
                "metric"
            ]
            == "centered_rank_for_90_percent_energy"
        )
    ]
    .iloc[
        0
    ]
)

raw_rank10_overlap_row = (
    writer_subspace_overlap_df[
        (
            writer_subspace_overlap_df[
                "representation"
            ]
            == "raw_dinov2s"
        )
        & (
            writer_subspace_overlap_df[
                "rank"
            ]
            == 10
        )
    ]
    .iloc[
        0
    ]
)

projected_rank10_overlap_row = (
    writer_subspace_overlap_df[
        (
            writer_subspace_overlap_df[
                "representation"
            ]
            == "notebook27_projection"
        )
        & (
            writer_subspace_overlap_df[
                "rank"
            ]
            == 10
        )
    ]
    .iloc[
        0
    ]
)

raw_global_direction_generalizes = bool(
    raw_selection_mean_alignment_row[
        "observed"
    ]
    > raw_selection_mean_alignment_row[
        "null_q975"
    ]
    and raw_selection_resultant_row[
        "observed"
    ]
    > raw_selection_resultant_row[
        "null_q975"
    ]
)

raw_centered_subspace_more_concentrated_than_null = bool(
    raw_centered_top5_null_row[
        "observed"
    ]
    > raw_centered_top5_null_row[
        "null_q975"
    ]
    and raw_centered_rank90_null_row[
        "observed"
    ]
    < raw_centered_rank90_null_row[
        "null_q025"
    ]
)

projected_centered_subspace_more_concentrated_than_null = bool(
    projected_centered_top5_null_row[
        "observed"
    ]
    > projected_centered_top5_null_row[
        "null_q975"
    ]
    and projected_centered_rank90_null_row[
        "observed"
    ]
    < projected_centered_rank90_null_row[
        "null_q025"
    ]
)

raw_writer_axis_unusually_important = bool(
    raw_writer_overlap_result[
        "writer_energy_fraction_along_script_direction"
    ]
    > raw_writer_overlap_result[
        "writer_direction_energy_null_q975"
    ]
)

projected_writer_axis_unusually_important = bool(
    projected_writer_overlap_result[
        "writer_energy_fraction_along_script_direction"
    ]
    > projected_writer_overlap_result[
        "writer_direction_energy_null_q975"
    ]
)

raw_rank10_writer_overlap_unusually_high = bool(
    raw_rank10_overlap_row[
        "script_direction_squared_overlap"
    ]
    > raw_rank10_overlap_row[
        "random_direction_overlap_q975"
    ]
)

projected_rank10_writer_overlap_unusually_high = bool(
    projected_rank10_overlap_row[
        "script_direction_squared_overlap"
    ]
    > projected_rank10_overlap_row[
        "random_direction_overlap_q975"
    ]
)

compact_writer_conditioned_subspace_supported = bool(
    raw_centered_subspace_more_concentrated_than_null
    or projected_centered_subspace_more_concentrated_than_null
)

writer_preserving_axis_removal_supported = bool(
    raw_global_direction_generalizes
    and not raw_writer_axis_unusually_important
    and not raw_rank10_writer_overlap_unusually_high
)

geometry_audit_verdict = {
    "notebook": 30,
    "raw_global_script_direction_supported": bool(
        raw_global_direction_generalizes
    ),
    "raw_selection_mean_alignment_to_fit_direction": float(
        raw_selection_mean_alignment_row[
            "observed"
        ]
    ),
    "raw_selection_pairing_null_q975": float(
        raw_selection_mean_alignment_row[
            "null_q975"
        ]
    ),
    "raw_selection_resultant_length": float(
        raw_selection_resultant_row[
            "observed"
        ]
    ),
    "raw_selection_resultant_null_q975": float(
        raw_selection_resultant_row[
            "null_q975"
        ]
    ),
    "raw_centered_top5_energy": float(
        raw_centered_top5_null_row[
            "observed"
        ]
    ),
    "raw_centered_top5_null_mean": float(
        raw_centered_top5_null_row[
            "null_mean"
        ]
    ),
    "raw_centered_rank90": int(
        raw_centered_rank90_null_row[
            "observed"
        ]
    ),
    "raw_centered_rank90_null_mean": float(
        raw_centered_rank90_null_row[
            "null_mean"
        ]
    ),
    "projected_centered_top5_energy": float(
        projected_centered_top5_null_row[
            "observed"
        ]
    ),
    "projected_centered_top5_null_mean": float(
        projected_centered_top5_null_row[
            "null_mean"
        ]
    ),
    "projected_centered_rank90": int(
        projected_centered_rank90_null_row[
            "observed"
        ]
    ),
    "projected_centered_rank90_null_mean": float(
        projected_centered_rank90_null_row[
            "null_mean"
        ]
    ),
    "raw_centered_subspace_more_concentrated_than_null": bool(
        raw_centered_subspace_more_concentrated_than_null
    ),
    "projected_centered_subspace_more_concentrated_than_null": bool(
        projected_centered_subspace_more_concentrated_than_null
    ),
    "compact_writer_conditioned_subspace_supported": bool(
        compact_writer_conditioned_subspace_supported
    ),
    "raw_writer_energy_fraction_along_script_direction": float(
        raw_writer_overlap_result[
            "writer_energy_fraction_along_script_direction"
        ]
    ),
    "raw_random_direction_writer_energy_q975": float(
        raw_writer_overlap_result[
            "writer_direction_energy_null_q975"
        ]
    ),
    "raw_script_shift_to_writer_std_ratio": float(
        raw_writer_overlap_result[
            "script_shift_to_writer_std_ratio"
        ]
    ),
    "raw_rank10_script_direction_overlap_with_writer_subspace": float(
        raw_rank10_overlap_row[
            "script_direction_squared_overlap"
        ]
    ),
    "raw_rank10_random_overlap_q975": float(
        raw_rank10_overlap_row[
            "random_direction_overlap_q975"
        ]
    ),
    "raw_writer_axis_unusually_important": bool(
        raw_writer_axis_unusually_important
    ),
    "projected_writer_axis_unusually_important": bool(
        projected_writer_axis_unusually_important
    ),
    "raw_rank10_writer_overlap_unusually_high": bool(
        raw_rank10_writer_overlap_unusually_high
    ),
    "projected_rank10_writer_overlap_unusually_high": bool(
        projected_rank10_writer_overlap_unusually_high
    ),
    "writer_preserving_axis_removal_supported": bool(
        writer_preserving_axis_removal_supported
    ),
    "proceed_to_low_rank_nuisance_subspace_method": False,
    "proceed_to_raw_script_axis_removal_method": False,
    "subspace_rank_selected": False,
    "nuisance_transform_applied": False,
    "verification_after_nuisance_removal_evaluated": False,
    "parameters_trained": False,
    "selection_used_to_tune_geometry": False,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
    "scientific_verdict": (
        "Raw DINOv2-S contains a reproducible global "
        "Arabic-to-English shift that generalizes to unseen writers. "
        "However, the paired residual geometry is not more compact "
        "than the mismatched-writer null, and the global script "
        "direction overlaps unusually strongly with leading "
        "between-writer geometry. Therefore the evidence does not "
        "support either a compact writer-conditioned nuisance "
        "subspace or a writer-preserving script-axis removal method."
    ),
}


with open(
    REPORT_DIR
    / "paired_script_geometry_final_verdict.json",
    "w",
) as file:
    json.dump(
        geometry_audit_verdict,
        file,
        indent=2,
    )


print(
    json.dumps(
        geometry_audit_verdict,
        indent=2,
    )
)


if (
    geometry_audit_verdict[
        "proceed_to_low_rank_nuisance_subspace_method"
    ]
    or geometry_audit_verdict[
        "proceed_to_raw_script_axis_removal_method"
    ]
    or geometry_audit_verdict[
        "subspace_rank_selected"
    ]
    or geometry_audit_verdict[
        "nuisance_transform_applied"
    ]
):
    raise RuntimeError(
        "Notebook 30 negative geometry decision was violated."
    )

{
  "notebook": 30,
  "raw_global_script_direction_supported": true,
  "raw_selection_mean_alignment_to_fit_direction": 0.524339564329646,
  "raw_selection_pairing_null_q975": 0.45241678528641877,
  "raw_selection_resultant_length": 0.5418086647987366,
  "raw_selection_resultant_null_q975": 0.46838934198021887,
  "raw_centered_top5_energy": 0.45087203976152895,
  "raw_centered_top5_null_mean": 0.5145750665763087,
  "raw_centered_rank90": 39,
  "raw_centered_rank90_null_mean": 31.78,
  "projected_centered_top5_energy": 0.6215773338127311,
  "projected_centered_top5_null_mean": 0.7903598140061866,
  "projected_centered_rank90": 25,
  "projected_centered_rank90_null_mean": 11.008,
  "raw_centered_subspace_more_concentrated_than_null": false,
  "projected_centered_subspace_more_concentrated_than_null": false,
  "compact_writer_conditioned_subspace_supported": false,
  "raw_writer_energy_fraction_along_script_direction": 0.05452933271817767,
  "raw_random_direction_writer_energy_q975": 0.00

# Final Summary

Notebook 30 performed a paired writer-conditioned script geometry audit to investigate whether cross-script variation in DINOv2-S representations contains a removable, low-dimensional, writer-preserving nuisance structure.

The motivation came from the observation that explicit script-adversarial training (Notebook 29) failed to produce true script suppression. Therefore, instead of adversarially removing script information, this notebook investigated whether the dataset's paired writer structure could reveal a more direct geometric solution.

## Representation Audit

Two frozen representations were analysed:

1. Raw cached DINOv2-S/14-Reg features:
   - Dimension: 384

2. Notebook 27 standard writer projection:
   - Linear projection: 384 → 144
   - L2-normalized embeddings

All analyses followed the writer-disjoint development protocol:

- Fit writers: 145
- Selection writers: 36
- Writer overlap: 0

No parameters were trained in this notebook.

---

# Finding 1 — Global Cross-Script Direction Exists

For each writer, an English-minus-Arabic centroid displacement was computed:

\[
\Delta_w = E_w - A_w
\]

where:

\[
A_w=\frac{p1_w+p2_w}{2}
\]

and:

\[
E_w=\frac{p3_w+p4_w}{2}
\]

The raw DINOv2-S representation showed a reproducible global Arabic-to-English direction.

The fit-derived direction generalized to unseen selection writers:

- Mean alignment: 0.5243
- 36/36 selection writers had positive alignment

A writer-pairing permutation null showed that the observed alignment was higher than mismatched writer pairings.

Therefore:

> Raw DINOv2-S contains a stable global cross-script shift that generalizes across writers.

---

# Finding 2 — The Residual Geometry Is Not a Compact Writer-Conditioned Subspace

Although a global script direction exists, the residual writer-conditioned displacement geometry did not show evidence of a compact nuisance subspace.

The same-writer residual spectrum was compared against mismatched-writer pairing null distributions.

For raw DINOv2-S:

- Centered top-5 cumulative energy:
  - Observed: 0.4509
  - Null mean: 0.5146

- Rank required for 90% energy:
  - Observed: 39
  - Null mean: 31.8

For Notebook 27 projection:

- Centered top-5 cumulative energy:
  - Observed: 0.6216
  - Null mean: 0.7904

- Rank required for 90% energy:
  - Observed: 25
  - Null mean: 11.0

Therefore:

> Same-writer cross-script residual variation is not more low-dimensional than random writer pairings.

A compact writer-conditioned script nuisance subspace is not supported.

---

# Finding 3 — Script Direction Strongly Overlaps With Writer Geometry

The global script direction was compared with between-writer centroid geometry.

For raw DINOv2-S:

Script direction energy inside writer geometry:

\[
0.0545
\]

Random direction expectation:

\[
0.0026
\]

The observed overlap was far beyond random expectation.

Within the leading writer subspaces:

Top-10 writer subspace overlap:

\[
0.4776
\]

Random 97.5% quantile:

\[
0.0522
\]

This indicates that the cross-script direction is not independent from writer-discriminative geometry.

Therefore:

> Removing the observed script direction would likely risk removing useful writer information.

---

# Finding 4 — Notebook 27 Projection Changes Script Geometry

The standard DINOv2-S writer projection does not simply remove script variation.

Instead, it changes the geometry:

Raw DINOv2-S:

- Strong global cross-script direction
- Strong unseen-writer generalization

Notebook 27 projection:

- Global script direction no longer generalizes
- Same-writer local script variation remains
- Directional structure becomes less globally aligned

This suggests that metric learning modifies the organization of script information rather than simply suppressing it.

---

# Final Scientific Verdict

The hypothesis:

> A compact writer-conditioned nuisance subspace can be extracted from paired Arabic-English writer displacements and safely removed while preserving writer verification.

is **not supported** by the observed geometry.

Specifically:

- Low-dimensional writer-conditioned nuisance subspace:
  - Not supported

- Writer-preserving script-axis removal:
  - Not supported

- Nuisance removal transform:
  - Not applied

No rank selection, projection removal, or verification experiment was performed because the underlying geometric assumption was not supported.

---

# Research Direction Update

Notebook 30 changes the research direction.

The evidence suggests that script variation should not be treated purely as removable nuisance.

Instead:

- script and writer information are strongly entangled,
- cross-script variation contains writer-related structure,
- explicit invariance objectives may remove useful identity information.

A more promising direction is therefore not:

"remove script information"

but:

"model the cross-script relationship itself."

Future experiments will investigate whether the transformation between scripts can preserve writer identity rather than attempting to eliminate script variation.

---

# Reproducibility Status

- Geometry computed: Yes
- Selection writers used for construction: No
- Parameters trained: No
- Nuisance transform applied: No
- Verification after removal: No
- Monitor used: No
- Validation used: No
- Official test used: No

Notebook 30 is closed with a negative but informative result.